# CIC-DDoS2019 LightGBM CPU baseline

Production: full natural-distribution train split and exactly 100 boosting iterations. Resume/checkpoint state is synchronized with S3.


In [ ]:
from pathlib import Path
import base64
import hashlib
import json
import os
import subprocess
import sys
import time
import zlib

PROJECT_NAME = "Luan-Van-LightGBM-Parquet-Github-v2"
SESSION_MAXIMUM_HOURS = 12.0
SESSION_STOP_BEFORE_MINUTES = 30.0
os.environ["PIPELINE_SESSION_DEADLINE_EPOCH"] = str(
    time.time() + SESSION_MAXIMUM_HOURS * 3600.0 - SESSION_STOP_BEFORE_MINUTES * 60.0
)
os.environ.setdefault("MALLOC_ARENA_MAX", "2")
PROJECT_DIR = Path("/kaggle/working") / PROJECT_NAME
SOURCE_DIR = PROJECT_DIR / "source"
PREPARED_DIR = PROJECT_DIR / "prepared"
RUNS_DIR = PROJECT_DIR / "runs"
for directory in (SOURCE_DIR, PREPARED_DIR, RUNS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

encoded_files = json.loads("{\"checkpoint.py\": \"eNrtPWtv3EaS3wPkP3B5EEJmR7QdJ4vF5OZwsWMnweZh2M7uLXQCQQ17JK445BwfthWt/vtVVT/YL3I4Izl3B5yBxEOyu7q63lX9cBiGz5usvTptsw0Lfiwur7rvnv0UPKvrtmNNsL5i6+tdXVRdG2RVHrxjTbEpWB5kXb0t1sGbp0F7U62TMAw//eTTTzZNvQ3SdNN3fcPSNCi2u7rpoGdVd1lX1FWLrcTbdftO/b4CFMriQj3/o60r9VDWl5dFdame61b9bK/6rijVY1dsmXro+yIXKOVZx/CbREg+L6jHb3XFRMNd1iEest0reBRfupsd4CA/fFPdLIKfsh2+WwRv2H/1rFoznNynn/z4y3ffvXgdrCTeySXrfoSfrInStMq2QJiYt8zZJui7dVrV76M4OP23oO2a5aefBPCnYUDCSiGaYBOJawJ94qRo603dbLMu0qC1V9kXX/0p3RQli3AuS5rCAvjYV9fpxU3H2mUA3ATsnjz+4svgc/rLGjsvLlmLTQRXEg4Vx8HP74vuigiV1DtWRWFzEcZB1kLrKi+ZgEENrwCN4KKs19fBciW+Jw3L8kjDJ9Z6DKMn/Q7nHlHv2KQJb3DFPvBf+vzXWVVXxTorU8QdSHBT1lm+RH5ZkwR+1TkI8opkLcn77a6V7RfwtUUJztp1UaxeZmULotIC49NrdtOu3jY9PrNd1oAeNO0qChfhIgiXYRwnHHAU9t3m9M+hibpFUIFD7J8LV7EU0UsRPWM2iyCH5kVFWsW5TBP8GeRDsnFokACmrOqS7XVeNBF/kNNgH4q2S+trehTodgzFPGtugDw6GGR92vabTfEh0t/zV8EfgzDptrtQFxQFSkjL+3DBaQ+asZJE8oqP4svAFt7Gz56iymFWqy9sRsUDRCGB75sCJCv8zyp0v23KHuRGe1+3yQZNXCQbgExXdRSLJvC5YbsyW7NIzdTgjcvQK6B33dxwnoqHpbIhZ8KqnIGoLpDT5+cLIkVq6HP7Tnu2Oe9ITgmjyKFiDZyYhIR2HzFRMHQZUS/9AgKepMzRIoJRQhRxzucA6+xcfK8b0Jx13eTA3UCSauANfgc+40fezDIlxYY+g/PBJtpoZjMTlQToz6o8gp6HyvEiqNj7sqjYKhwziSh6DSdX8m2x7v5GLyIp2AMaq+FnbPfnEnwFhhS6jnxt6vetYvlHEnPJXk3G32VlgYZbSvlMAV/3DQpbiogLkwZeypZr9bVFIYEGEWf7Wai+hOexX2yESLEPO7buyO6TUjRZdcmiJ2g+usjBIgaJfSJpANKkIfCHlYKl8bfJipYFf83Knr1omrqJTEnbhAKbBFUw2PbgZtd11WWAJ/5dVH3dt0FfFUArfbQnSXLrYHf3dRDa8OuLljXvYH4oHavbAcTZ8qvzuwBGKo23p18tz+80KIKV6zJrW4ju3gCykvoQ40G4l+XZDkW4bzEcGixAv+MmmkeIawIPfK13ICdD5Mjfy3CRe6kNRIxFVXRpGrWs3GCnalNcLgNHUFDjsouS5WkN4JoiZ8vgoq7L4J8kJcBT/MuWGrJwBBJ8BjIeu0T8zVkoIILcDK0RjUR8QF0dOoMM2CgERctHZ+CGOGy7iQ26AVUoPLhwGqbys4vTNvsAX7umYKgB8KQkV0DQGkBvd9wO/M5F1rK0Bf2ocoSygTEHAG4TF4uLfn3NMD4Eq8Cqd0UDHIUYNwI+KTi8TQqfoT+ERmGcwOdiF9nAdg1DzzAJjLfxARN/h490V96wSxCyaZC8zTR+wGx0Hbyp5TPmjLHJyvIiW1+ncwYTDBJQxQ8wZD9TcmKTrC0uKzA8EAuvu0E/TD0YeqkOZK1dpEGv01evX7z54bufX3ybPv/l55c/fJe++ubt9+E4ZUyYFnXIY2JkEpnNYu4858V//jnLgB01RTir2OwzqGuqJBXZYsKRImqIt9Vdyaave/tUfAcIXjEUhNJVBg2h/vyHlYuthwKOVwlfKXLI/mRYAwE3LzYb1rQBZa7A3Ge/Pv/Li7djyIlpKuTEs4kcf3kv5ARcGzmQvJc//IeNnGlpHCp5GituOVhb+pOuywIcKeVRYyojaSO9ABInQmug4wXqqV7xkexkFqK3NrtE8KEolVyB0hW/cYIIQ98OLKJxBpr4+SX9wygzXvcV1go4OwQKFnl5kSJ5nzUVKGIUnrRfB5BqZ6VR8mnYFkMT6SMXgR+Y5S8pIZP+/d93Deh8090M7p6TnxSKfDWwYelSXnBJuldrsp2RCKi+vDpzUXf1U/Mr+7Bmuy74gRoQYdDmwNt5RAwJJOKifPf7K1YFHqZCG0mvmMs4jOKhl5zfiqObCKqAXQE6C4+B4f9K8wyGk6OKgg5KD6mA6v+A4BTTYBFVNawEBN9ByATGyiqGcID8O3kf/jMp/e6VRt6Et5rg3z26lb3uQtuyUGQkP+tYNn2lYwhPRU74zUF3mL8+WUCLw9EwkvP47NFn8V0YG6EnBTwCARRUkX0QDmvhwIf6kSmqECRDiIMCsgxekIChAIxYFMxKsg4DZspGVeLhBHaYcsySdkECiWUUe0V+QGxU4Id5BCtXWoVSStz/1cHYA5GTt8ywMjAWen4efBF8/nkQScCnMG8/JNtYgdadtAqhk/zRSR5sILMBrYxO2vjrgEajcm0VnCRPNm2oMXche7q0X2CZl0VAgjiRtdoFn8gIalScbUvGdpHdDDIohtXrgbhgGtBfmHLBrc3QypJOSATbrh0zlq5gCBMoOw4f51pAn/WT0AwDiAI9hGXAkw5kukXX7jV8QloHxLRpDlFiXr+vKLTjGgkaPRgAIx+kUtE/bdcgaG6FjF66v8OIRYqnasvjYYkFys3tXUwvh3qQLAZABtx2WbVmEcEinGInT6BZ34YAIlwGomGI2bl8vHNoRK9HCLTLmi7liaKgUVv3zZrJyuCmqLIyNeimpcgadpCD/0pgIOYAqdn2HTot5CiY0zX4edQfHC3owL31l1dBJiK407LYFlhDefXLm7c8lz+C/lSlyd6LEA1TexBmVFgfQ7RZS54ABts2sulNDY24fRgjTpqJOF0RDgNf8CIaJHAlnMgJmoS70GPwRPWGzbHblB0JgHuWUEx7Cm69BjOBmEUeY2D/EaW8lrHr6HE83s4IJZTFiRMcLBrvxldRYIpnYd+U4fliummeddkK89VIdKICJ5YX9nTEUmS7ug3xb9CZSGOFXBGI7/bAQFtW993qyZ8fP55oqkIDl0Ttrq5aZS9E2BBqeik5A+Kpfsc+p80hJWRnU9CCFIxI17dRPOrhseR+rG+fkTQoq+9+sjyvv6+W8IEXIJPBVVU4ZZlZ0BeeUqD3Bh++hP/CEX5wNo989Lhpt+E4QUWGYr3l9kmkLciUYr1l3VWda4YY7BjwrK9yHF2L+pShNUJUJTRgxCDoaLATWH/5gVsyj4sUIkKm77V4+glAoA5pLin8/u3bV29Iep7XOQMbsloFXz7+EhNTNH22K9LBkmfXYREEqvdo1WC0x7chgMQvP9dv+vXVX9gNf+heIh3CO91R1RcUiNNiUevz4RaJRIlNTyD9DtTimJZguD7GgsAdi1RcT6hhenYNLeq5nCtFE3bCBaFZDooFFr6ofHuRZ0vXLmMHaUSp8/liMHBfPLbtqTs3JQnc8qS4Ci1E56jZ7rVlXjvm0tYwrTizlMsTyJtBDJ7uJlqL6BlVZFZadWYRgKSukLdzkJlnUFVtwjAD8WFE45Z20BlR9eeLtofEdA+gR23xG8oqLiEIt4oMpLp6it800lFdqO23KNraLhPeLX4A1bRCPjPaw1eKDIdpKwVyALsvO48VMOJp7nKCsYFk6VsDOD43i/BaJ7fdUCbcgGHVsiouFKfCnaoUloLnW4XmXXg/v4+aMlUs9EQC4w2nzIXIIleOTXMYGu560DCcYBhQBdLPW//bewTZKtCWw88JtA8Jtj2pcAKDSXs+TBuMOoXMcmuAHsKOxqkzY1U1yGKYaDwd+c6KV3m4jlKJtnmvExxxhhK38Q7SGygK2i5xoOGEY5yIE82JzJ47VgUqaaioJ9+k0coQq+oAydMfWXXZXQHxT5/EMaabaGon5Isr6Q+/OOsrNBAVodGO60vsIvQORxDds8XPT4pL1h3DUuh2BEeRYIKhA4BzqrGwbCs2JRk6cQx3AfR8wQabyzcz0mYi2R03U6RrzlmxrxG5sdL3V07wFkSGgO4xL+O7Ij0A3R2StJgnHPi9BO3N99+cgqAcImujSexhiewBTm0ioZ2d1LqJrTenBa9ebLL1AXnt3tx2dn47Id3jTvhfgmfZ+hrmjquz2x0w8KIoi+6Gr0+UuDv35rRo255XeLdZdwrmBcuBtLyMvqhNPnaVC3yS2ln0ESpdY45XjXmc852d/HmMpDn8hGQYDecLxUFuXMGf78ZtF753LvtduMGNPW587yRmzVu4b6vnvV34qFUdds8d6MJxW116kLhp/KHOkw7Z4Q3qybhICqZpcIFhgmor+shpRw3Ee57WDXz90+PHI7I8j9sGRWYx+8A49aAY9eHi00nEDxHqh4xJHzYePSAWPSwOPSQG/Tjx59QU5gnpg8ecM+LNebHmsXHmg8eY/9sWSe4ZNcrg0BMDHlRtIXKWhxVNlPTnrGTAe6EA/GncIe8P65fzNnY8r/syp4IaH1KrhA2+kld8STWIXr5q3eDCcUGX1nOHahkeQTm9xeOBCf7vSyqpf9CLaC6F/DSbYYNEdZcWL/fYHr22rXWb8Ly0L5UXYBeBUfs2CDARTr740DXZN80lLrCqFaZlcBtyIww/pTrfja2xHhWPzTPgdih59DKAQY54Fr7eYPBM+kvhLs+Vq8Q1NytwJP/qW7YTtD3cWm7Ct3rI6DGQvDJsTPfOZzRnEJ9CKrkAM19ytW4Tkuth1Hhj5KBS34l2z2HoN6QOIM18hHBp6kWIy5dLUz2mNg9I9n0LbmGNewlX4fNfXv09vJ8uHBp76hHn0TowtaoxJ26cln0tvvwYcv9SBgejMq+thvxfCBL+p0IDf1gw4d6EhIkY4B52dkZMsDceMDJmdO1EFnvUYW1VrvGLtUpjP4LnCPVC7W4UB8vEZvYH3rqg+pnbJRD5pWexfs9u/5cwNbkbQ6rLa7lJE8gk4iUclJasYfJPl48e3WrMu3t061Mc/wTuca78nmfLJTd1PD07YA9YgD5yf8iMVef93BtZgrVWgCUWwa+vf+SGbszETacDBxcoQ5cSs8IAfaeKmUQ/RPp8UInHf3B7ThXayLvVmL/PUo9xS8FY9j3n3oHZPtBYJjZM5ki44ZpV5QUwQqO8RKEV2xtwDsXdi7ffjwGJB34Ls+pjydCor8qiuo4Mr4GLIHR7gm8Hm/fI5bRfMA6K7bX5xm7E32nDm9i3LfZg+7ZrH7u7TdHyoC1uruGYv8Ht0F1pqj1xPPbu3zQ0BOuBkxvThgZz96UJVNTJ1jaSg5+Fz+r8Jjzn9+bEEIcZ18sY5/Wfq6N7P2VVdsnUhT76GXvzMKRGSqAuLi41dd2RtINoU0w0tNjCyCUFnDyAsjaH0cjp+Ol9beSnc5vNOfAvevjO/XONGyYGPehYsvbKOWOvJolH7NWD1cqZMEYzuLnd+WDDb59CU3GzQqQIsfBN1u6aUQaaEnS6RcSZ3qMgFI2aviKBDk3D1tYl/5gWuToYSGrHcsl1+y4F8+CdCGp4jxGFEp91+d7WnVKk9mlimlh8ox/icyYRx8oCQlNpVulcoWMnxVh4kpb/FMkn2YBQbGnGYy5ZA4KHe5lhmEr83GV9SyddQ1oJJ+sBTeumC+/8s6UzJjTOWcjpahwpl2ibvBvzTXzR3NP+iHPzPJ/ce1weEKSWH4lKFqVoKC+hrGOlg+bdpbfGNWh02n6Dj9FnJ38/2Z7kpyffn/yERzvts6WYlThnS0mmX5mXFuhHYEx94h11wC1Yd3wrnIrn7Cr/Ig+Rzrrahpb2lV7fmjSU5FqKYSz/J5m2FAPbnx0MwuXIdTd2T74EhHuToYu6qU5rpZ1dc68r49NZ+OTZoyCDXvt2hcNXd5+0BXQRzLQl3goB7s3dMuIqGxEa+3wHlzB1xFUIHO+l224EKa/ekJ3AUmsn7UN87iBgQQ3jOEjrPU6n4cYGeOeevCAzKw5YC7kxxnzkGy+OR6jvCfmPgr/QyDGUW+wiwWALVeOjrg7B3vtN4D3mSOd2yVYl3YcOpzfBYKvx9PQPRGqLJ5jX7SP9gikLHdGEUDGazUAkpbrrjcoh2kgAFvyck7IpN6BYKr3gvlNXY4PLqh2B9MSSPivLHct+bbSYpV2hJu7vc0GMEtjUYu30xIAML6bjPM5C/lZUy8+957PV+vm3giZgDkyM1TJ6XjOe0G2zbn0VeLVyFEF9sjqK8v3BSBoXoB2G4iAQpA9TlroDZ8HOzFR9we8YNF+eezL4392uU23A1Yo5dYF7GskZBpJGiOZI+cG3O4ne0+Mfan7GbiDkF6UJ8XVDonNPOk6NFxJNIwbMIKIY+DyeVOtxod6I9tN6vlzwi5f5Ha+OzVl6RVhrhzfSblu6/NaGvGEZ3crcAtrbzNukyxosXZiR6sIUeBnXajm3X9noXjndLkDvht+8SKH8jjUbSH57jNyMqstM7dMSbd58RAM1y88F120uJXoM+EHrG45oifN1grOJ8z2KZ0mvG637truQ+Q9WNnUcf0bXsDpil5BcU6PIqOHydxBOVP12wGA1hZNZfBcjy43gf5ywCnuuIuV1QzmXGVMNjVtOncozb7jQYMaurydvtTIkaNzHy07r9t14H/gYOsmT72rghYHEQodu+yJP5shdA1jPlud9T+wUj8cauKAMX8MSL4C/vNgm4gL40NucWlqxrNWu3oGSF78BhKVeoxvwAq7lfTn6fV/GqwwobfCQD3MyX+ed3cm2giJbtl87ybJme6GL9mQ39Fhi6OB56+eV2pmlx2ma9Nq9rFBt6Y3vULbikfGAjfyqdkEJjfXWaeEDCxNobeYVF4xQy1X46QjLqUwIvz4A9ZQjnDquqI+hgVS35aQNLqgTJXEfAnkC5z4mebOOB/GNIDEH497wu3z8NL+zbBiRg/7ZA9pqZXF/MQxonFM5ssqiAz4qH47njGIauANT3INGAKN5wABoo+fB12sa96i92FHvDvoaQhaZIjQasnoirOBURmDm5QN6tJFmG1riHg1s3Tonsy+hXUwHq+NrNXMDP8snH5Z/m755T9+P4aPd1Nm6UMGwyE5nZTKxmzKaD52THmUxDzQt/6/0nmo0uRe+k1FE4a7WGSmiP2fzLXGI/ZxTaQjWd7yhjre6o1/5y7dn8oh8m90EdVXeBBeM5oO3WATdFdPvUecjDDfc6xUobSPIiC3wCPQm1Kgm/ak9D9uVPsw/czInf3IyJ4fE00nK2PaYI3VPAzYuuhtDdm91QvBr8DwVE62RIdaQzqVyZ6t3xcy472YN2nFJHkTW89w9NketGYnts+MzvpVD21f+mWrqccx8UjltEyfUff8wC/65ZmwnLu9/LGpS3njzLKS10hTbG3GmUamiV6gmNXr2SI2fXJb1RWSEl587ERnucqvLnJ8tAzhny1Mc7Zz/wymAJi2w0yc7Ry9zbefUfwMwZq/q\", \"config/data.json\": \"eNqNVEtPGzEQviPxHyKrhwQFEsKjLRcORZUq9YBEb5BaE3t2Y8VrL7Y3ISX97x17HyxRkBrlMJ73zvfNvB4fDQZMQgCPgd0MXuO70XCpHKnYZAV5rnGiTFmFiVBCSutn0/OvpyW45wrDaYnuVGjwHj0bNxkypZGXEAI6E7OcnExOzpqAzimAyzFwYXVVRC9TaX3QxgUYqagrqnAzeGQ/YYGajQdMt8K3WD8KohXqBFH6VUvzNnUK4pmzBY9tGiiQq4wXyntlcioQXIWts4eipE9R8oNWOnus9JAegx938fFd201PJKnrIHe2Kg8nrD0GlLmNb6Oi7sFWTlCB+5j1Dn1QBoKyptE05nvrwr5Dq7t3NliqzOZ11vn7D/Wc0ExjafCI1r/Jh/lSqz5LggMVYZuefZ62Wdag47dQxWQ4v+rwpF72VB5Rkmo2nV13KsoZMN9GykAVLK8nBRtw2NHG4XOlHHLQmje84whiydsGI369vkuHpbMCW3jb/vGF/IUKXDrbopEw6GaSDIkfDZXfQcSGv3eP/OnJz0cZYTWs5VslR7ef2PiA19A7sfMJo1HrPVTlriRsRh8GSR928g3J/48MqqAwgvWw2atCaQ1uGQJ57LHBVAU6JbitAq09l2FbRkYw+lAIF7MOCwOGK5Px0tIoE2w0bA2CJma9CmpNMBnJDeaQHuSrjApbvlFhyWP0CrFMQmYd1ypfhnxRdPm9IEIl1JixBlkP1rq1Hp7Obmr60p2J+tn17PzyclwbhS2IB0SBxEz2xwfZT1ZgYd22l4wYk25Ys99d7mY1rqb06w5K0zU5wIoXlQ6KmIXxfF6cdV4FvHBYg6LzExMCXR8HoluVL9NeO1DJd6u2ALFCE7eFSSQiFjREYoTg1KbN3qb1TAuAfAGBtiG2vN9pRuW5pZvjrPf1vnC7Rqeh3Lt7rWdzqj70T00fH9H/H5XK51I=\", \"config/data.smoke.json\": \"eNqNVE1PGzEQvVfiP0SrHhIEJIRCKRcOrSpV6q29QWpNvJPNKF7b2N6ElPS/d+zddRcEUqU9jD3PM2/ffDwdvRuNihICeAzFzegpnrsbUZLjq2K6gapSOCVtmzCVJMvS+Pns/NOpBffQYDi16E6lAu/RFyddhBUpFBZCQKdjlOPj6fFZ9yCDArgKg5BGNXVE6UapV31Cgi6JWXGGm9Fd8R2WqIqTUaF643PMHw3ZG22AaP1srUUfOj0SK2dqEWlqqFHQStTkPemKEwTXYA/2UFv+FSrfoJL9MdOPdBh9+xIPX5XZDUy2MoPKmca+HvAuv1ssnnPwgoVOjBk3n81m0fsnYQpvFQ0LGBxQVHR29nHWR9mCimnItI7zyyw1+vDiyiOWKcv8Kl9xzIDVPlYTmmBE+xOwA4e5og4fGnIoQCnRtYRAkGvRE4zSDnhbh9YZib3yPX98ZLykIEpneqGSPFmT5Eil67osuVsnu8e/Dnfi/t4vJiuWc9zat1RObt/3ZJ+hxt7JgzeNkzjp0WOyB2tcmLz5qPThULJ6pJOu//8yUM3PuKyvuz3VpBS4dQiMaAH5z3VToyMpTBN4IkUZ9jZ2RME/CuFinmuhQQvSK2ENS5nKxmIrkKyY8RRoy2XSpdBYQTowljSFvdhRWIv4eoNok7EyTiiq1qFa1jm+l9xQqWqFNhqLQVlbaoN6OrNr25dXQLw/v7q4/tAHkqbmPuAWSJ1Z/PahHAarsTZuPwjGHZPWSzd6OfZgNPKod6TZDxtRNyoQNxbGxXZxllE1PArYAvFiiPGA94IDmSflejhp0JTPJm0JcoM6DkvhH7jH/81CexRLCNz/kSRjLgfcVpxQGJ5/Z7xvB0SYLToF9sUO6pHd2ngTn2gevePvL4dTxI4=\", \"config/orchestration.json\": \"eNp1kUFrwzAMhe+F/oeQ88KSdIO1x1166WCnXY1qC8drrGS2VDbG/vuUNBujsIPARk+f3kOf61VRlCfwvkdzwkTYl7uidEKevHwgtQ9N3Wy3zW0vQNVZqw++Y3+M1QjpTZArH7iTY3Vuy5uZ5oAhI5s8SLL4D84G69yQW/39gkZMle0hZ8wLakzDK1o2BHEGHSYTL1qHycT+8al6Xmb3VyYYklcPgTEBh4F0uqnrSy/bDp1o4BhIWJftis3SSmiR2IySO+MFkvujubtfNEIUyJsOIfERQZMyKK3TvJOu2Vx0Ed5DlGg0TlYHBpgxjjxLmvpKw+AJdHNCfaZZtGCW4/xQOEQcRHeiHcjNvjZtXa9XX+vVN8xHlw0=\", \"config/report.json\": \"eNpdj80OgjAQhO8kvAPx7AHx5+DLbNayhgbabtptMBre3VJR0R7nm+nMPsqiqjbsqdVKtLOgumh78G4Mm3PVHOv0ttlj8KZNNAkpYA88OIGAhgfKzrr+WunGA2oLKbKyrP4KHfJv0+7DmLyJgnmMJyaUHH5BcQzXJEVPAcTlFYnul+yFrOoM+j5pj1mZNRTVQdB3yjXNYbuAEVMRgxbyue218U0NYUgthqz8W+rZMi2XELX5/OZUFlNZPAHro2Ni\", \"config/train.json\": \"eNqFVdtuEzEQfUfiH6o8E0hSioA3Lm1VUaDiLiE08u5Odk29tutL2lLx78zYe2OJhJQoypwZj+fMmfHd/XsHBwvrzE8sA2jR4uL5weI8Cr38Qt9zWTfh9OXb5YVwVxHD8lSGJhbL3WbxIEW2pkI1xCl2r4u2A/HGopMt6gDOqORRCI9KaoTS6EDGztMjVgRvVpsn2VDhTpYporSxc9KxhcIYz6dFzf7r1arLJJy6BR+MtVLXhGyF8pgx2RZCCV0iNEJXKuMLbTR2525RhOgQ6GZEgjR6hkePIJSC4ITUlPvak0NwEacUWOFEy8AdG8mcbkrJINzaVEhdVCGfSKgpmHC5S0gbVZClEt4PuKKKNEc7Edhn9XC1WvcgE0EOO+R8h4O5FTdQoQ0NWZejle5ciSCAfilom2ge0oi2qASodU4xN2+S+a+jauYgGPBWyfB3FOcvJLO3OTrqjT25Wyd6btdjSCHqmqv8H4pXnGpIhMHJkizfM3WgTK0Ms9dxCeiccYsffUBW09CJQVIdl6FxKCom8/EYEdBRvZJ6WE7bzSUZV7KAFVxLj3tBEkkPjkrco/NJkXugOXf7fFJrHQnbtPvghgowNUkTrKELe/mL77TePB1ZZnlf0cAHwiog32pWElPE5uzFLWaq1kOObO6mAzVe9zqbHrJDVxgvwy2Hsu13N+h0f49hMjjWIQ0TXSWVVknHLTMx2Bj8I7YNvSNZRMmDK7YILbbG3QKtsq1U864oI6pEcuB5qvkSC5tXGrFGP7QeFg/6HuX/UIhQNv28P149ezJ34C7XtIsslKJsEGjROZmGct25knLJTVynWro7jbVTTHlpjdTT8ukfup1Qecmlw4ZWeZp5kG0bgygUZg/gcv0/9ZZ0wiWinTltJtR79D7P3N04wbKlbjcmupR5M8qElysUSPomrqWOIW+f2fznS9NESQ/j+ToqNU18OMmJmmuZay5abhl0HZ6jKRP2ZB+NegikAH5iKDc9MIm9SQlFLC+p4ah33P+Ph/Dy86s3x58GPZHytvJmgl98OD45+zbRW81T2OEvvn6ED8enZ+/fDQ4066oQ5SX86/n6+OTF5/NPfcSEDVpedX6U7oblv0PFgWfvTt6PT0L3uAI/qqYa5uj+Pfr8AawiR78=\", \"config/train.smoke.json\": \"eNqFVdtuG0cMfQ+QfzD0HDWS3ARN35LWNoy6SZBLW6AoiNldanequXkusl2j/15yZm9ZCyggQRAPOSQPD2cenz87O1s5b//GOoIRGlc/nq1ukjDr3+h7I9suXr37df1R+NuEcX0lY5eq9XG3epEjtW1QjXGK3dtK9yDeO/RSo4ngrcoelQiopEGorYlk7D0DYkPwbrN7XQwNHmWdI2qXeieTNFTWBj4tGfbfbjZ9JuHVA4RonZOmJWQvVMCCSV0JJUyN0AnTqIKvjDXYn7tHEZNHoMqIBGnNAk8BQSgF0QtpKPddIIfoE84pcMILzcAjG8mcK6VkEB9cbqStmlhOJNRWTLg8ZkQnFWWtRAgjrqgjw9FeRPbZfLfZbAeQiSCHI3K+89GsxT006GJH1vVkpZobEQXQLwXtM81jGqGrRoDalhRL8y6bvzmqZQ6iheCUjN9Gcf5KMnu7V68G40Du3ouB2+0UUom25S7/D8VbTjUmwuhlTZY/C3WgbKsss9dzCei99au/hoCipnESo6R6LmPnUTRM5naKiOipX0kzrOfj5pasr1nACu5kwJMgiWQAJyWe0PmsyRPQkrtTPnm0noRt9Sm4owZsS9IEZ6ngIP/hmra7HyaWWd63tPCRsAbIt1m0xBSxuXjxiDNVY45i7rcDDd4NOpsfckRf2SDjQxYnG//tN50aCBhnm+M80jZRLbm3RnqemU3RpRhesm0dtD3gOEJSR5K8v2KPoFFb/wB0o+2lWg5HWdFkriOvVcu1rFy52Yg8+qFbYvViGFX5D5WIdTes/febN6+XDjzslq4kB7WoOwS677zEIqd+kB7JTdzljvqaJgYopj44K82cBPqH/ihUuevyYePEAq0+SK1TFJXC4gHcbnjSb00nHBDdwmk3G0DAEMrqPU6LLDUNvbPJ58y7SS18x0KFJHPiWpoUyyW0uAZK0bRYMsB0/nYzT3s+y4iGO2mW65IcTwz6AT+BcyacyB71EEkB/NJQbnpnMnuzAqtUH2jgaI48/8/n8O7rT79cfBn1RPrby/sZ/vHTxeX1HzO9tbyMPf7298/w6eLq+sP70YGKVJWoD/DU8+eLy7dfb74METM+6A5ry9v0OL4BR1QceP3+8sP0MvRvLPDbavMrmE95/ow+/wHDxUlr\", \"data.py\": \"eNrdPV1z20aS76nKf8Dh4Q5wKFq2s94sd+mUbNNZX2TJKzmX29LyUBAJSliTAAOAlhSd7rdff81gZjCgKMe5h3OVbRKY6enp6emv6WmGYfi+ytZplQXLLP2YXmR7dbrIgldvX+29fl2ePt1/8qfgfVr9ssmaoF4v86YOFmUVHOYXl80PL98Nv/7q668+XGb8LsjrIK3r/KLI5sF5Bg2zYJGlzQb+n5XFp6yq87IIjP7B67RJa4A9q6AdvASAJ+UVgIEuRQY9gvN0mRazbD4I6nS1XuKHqwy74ycAVZTVKl3mv2bzYRAcptVFFuTFetMQjK+/WlflLKtrQKgsMj2VqrwKLqpysw7SJkiDJl9lQVrMg6sqb5qsgFnwjPbqdTbLF/ksABo1NWAXhiHOeVGVqyBJFhucXJIE+WpdVgCqKMqGJlJjK/W0uoDudaYfXMz0x8u0vlzm5/r7P+uy0F9WaXOpv5S1/rhepg1Qd6UfVC3s+hfAO3vWfm+qzazRX3Gqgv6sXC6zGSGr8H9VboomqwbBPFukm2Uzz7ErtZ6nTUZ0kqbq+4BA/grUlYZrQBpmpNq9pznQm+ZmnRcX6sVBcTMI3sJo6fkSoLxL1/h2EJxmsESw4gYBi81qfYOLUqxbGsB6pchvwXrePrxJK1xbfJq6T4drWX18+0v7tt40+RJH+/qr0/eHbz8kRwfvJqfBOIjCpkrzIhwE4SdgsTktLH5rsroJ46+/endw+uPzb6FlsR5u8qJ5/m20f/3G+QPtfpgcTU4OPkxeJ6cH794fTpI3b+GfV8eHP707gt5hwqydLHL4J5+Hnh4nxz97OsCsuP3k6NXxa2h9ePBycmg2XKbn2TLk2c2WsDsD3PC8KYDc79NNnZ0gxWvYUNEJLD+s5gTIVcWjr78K4A9w/Ema4w5KF7BasF3mG1qzoC431SzbQ6yDc+CceVrdBFeXsH8aEAk/phcX2AgHgl0PsqHI0mp5E5SwrYeyj77+CjgtWJbpPAEJscgvImSfEXJt8N/EO3Gw9yJAPjyDZwPkmqkgxh0S7ABTxbbUOea3Vzk8NpoMy3VWRGEF6wfcVc5h8uNw0yz2vgtj5IhL4KdlJqDxT76wutebxSK/Hs5AXC3K5TyKgzHQd4j7NTR6tYgBTvhyiJOLGHrctsuWdeZ0a6ob5wmhwVx6k66W9svsepatm+AtvacVw2nAUw+QChcwMFc3Cv9+8O5QUIX1RNYGMfLLJq8y4JEbfPvn4N9Pj49g2bI5LF4JsGE/gGgAQs6BhjdAONrXMGYPARDrIWqUpEsFoC8IS+CLvKgbFPERdxvQasfGLBj7/0iXG4X7KxvtEuCsNnUDSgdEcFCe/xPkWijjyKTmgM5tOGd9g5uYBDx+WJsbAh+UmwYUCH5aZauyusFP6WYOre8Y5CqntgCxBtrDvlFjDOf5YpFVWTubuJ2t9No2sUX4TkDbC1OLnB4FtwLlTk8PW9SAytkCaNzIsGcyvelZka6yaUxaGz+CcgwMKTfV2KXFTfQJMQn+Mg72qT1/hQ48RszqltXSMK9ny7LOonqziuT9IHgy3B8E6XmdNOVy/CTbe/J0+zqSfH3cCtfHKFnVlNSKrss6b/JPrKFhuKApgydhS1c1Y3sdp8OLrInCegbA4Wsc/Ats1wLUVLgVow+XIKi0eXIOzALdM8YF547wMlRIWQVWgVg3tYENqAG9BsJI07MQBHWdrLMqQTsihPVAIm9FhPsOrY4dkrQ8DmgUQolWrqZNucpnCYqhZA46FOTjDW7EkdK2rUxFlV83eUHLMGol7xFQTPA0GqAqzYpmuPo4z6uIv9TjD9UGFHl2nddNUn6kr4Jfk6GUQvUwtsCgkE5Yskbmc34UfAMCtlmtQ1Oka1Ai0K92FugkjU0yDKQNAqjRjkvrWZ6P36QgmwewkCDkmvHTAW3y5GN2U5tTwj/cfYhWYxaF/ygUomU9BF5cpiAFNLoWgeN2jeawkVAjJmKdkBFQRyioEiCuqQoHaFuBCi7oIS3PEmh9hu+UUiRZKNpQwbAFLrYY0iLVUXd3voHRj8rmDapzJZOUkQ6gQA6BQAzmZVYTMIIDUgmBapFEM2iFIylolCf0IWd5PbxYlueRzCdG3NYsVWj+UWzjTBB3wvWo1EY+owHCanYJ0v9WxvqX6i6A9mDI2EjLFqJO7erkBYhzME2Wm1UR8X8ghJWJitsHdo6S1dlcrRbuGngO7IGCLXO70NK1LWViaFawmoJXMlhsGBwEXD0ngvJnJKmgducIRcKpJdsCKQUD8EgkIBmkamsOF1umEHcF2WggvEWNvdIgFZK37Si4BFcpMxDBDY2x1Drgc0WXqiUlTbal6/1zU213mBri45teByN5ckROj2IV0BSwOxJyK4Vj6n6W8bGG1Wba7m/8+sV5xKJqgtu7j7LprNmkS7IwbNqiPWGR1bIyLOBTi+ICEfV5ulxG3KOlvwWG28b+NTnTaHj6TK3FOpsaS0WOi3K3wHoDfxq0aaJ9D6I9aHIZtY0xABWs1lrOh//4B5qJjx1xAjCGaCMn5zdA1Ei8/eH5Mv2YPT2PjOAF6TCAIxoMreALUBhJDW/H38VD/hrBi/A8vwgNDZKQqbfKr8H9JJMNeAocUnTHqvSGptJ+lRn9ChPhtsO0Br88i7QLi/JsfWOqOtK78D6rKrDUQdWhvhqH+QWgDyaIsTQINvo1+C/858ULwy1+th8DezyyHOWXb/7w3bd/fP76yavJt5M/vPxTvAOcp3/swvnTt6/3v/3Ty5dPnj178uTJ5GVXmPhReoKg/jVgJ95gDqRmgosBugiXDhdN1s4m7UAsVWMLkw0O+7vOUBnAQH7yw2IRL5lLx/D7loOMkhim0U4Ax1CibFPkqPYJrgGC0Hn+bRw8Dtg7eProEXxVahqWEyUmveGpnO1P5S24HCULVGr2jd3sydTmcxgLHP8qiwiRv3CnQQDegPuG4YKrMAiexrGBK0zqO4OrOZBIEQ5jRSLZs0TbQcDxjy+0KDKVvvWXsR6wQIJrrFBiDIw5sq4Q+IsKpNcoWM+HaG+9wW+DwNImPiuiMwsQu2WRz0hgE8jhspydjQakSiILXjxVkwkBHnlJQ8B5WaRR+JejgxeOLAPMMFI2RHwTDr8l7GpHelA2m6+VDT0DaXFRVijCSJ4MmzKhYF40x3HHmlamONPAOHjFMm2EXoqym0bazAGc8hrwpTbbfU3lWTcYI24Cgk2eA4U8M22E8IsxKVIGO0TqrCPbJKVmWwecrNbNzb3DCXHpvRmmm1yDFjvkoPwBBh/Kqg3Ivc7rj3vn6ewjSJEMGwYUnwhK4KmsyIDd4MWTp9/tncNDjhUGb1/XpHE57i3SRsJw7N4B7ZMcdmmSgGxZLgYY403RBRbFyD7IORrUuC/rdi85VpPV76G+otDYhuHxV7oDbQrw1T9GBhycxhDkTcHxE1xVDo+rh5EFob8nIJDNNujlvT85+OHdQfBPMCEKYNEVyInxzweH4QP61jfF7LIqi3JTj4+OT97t1tueefjqZHLwYRJ8OHh5OAlydFXzJgd/J/oIOzD4MPnPD8HRMfz96fBwoN7fBC8Pj18az/nY5u3Rh8kPkxPjeegM9v7k7buDk78HP07+TvBbiKBCf3774a/HP30ITo5/fvva6Pn5c5q8ey8TI7ZOiOWCyJ6FgdPuOLTMC7yAoZr2gc1+bmsnXuPf8YTtkFksMTr3hW00WhLOn1VlXbO6A/T2nUYiu7e2EUDzDbxH4VsnaLzlhdnB2O7pfM54yobHtSUDeGCw1Eif0pyRFTsdGDqyVwbk2AeEFtIZPkYtPGPyV5d4doCb36EtITVi96eByaihp+iGTHeJm6M/kHDwsrjIImc9Y08PPe4wXWOIO4qK7LqJ1ETigek3GhH406ZcE4UodNaFuwZxbj8VFcKT7HY4r0Ds2497RcrryeEEtsybk+N35mYJ4536rzDqG749Op2cfAiOT4K3P4A8mqBAODah6Z0XB/9xcPjT5DSIvo9DUQXOSMSfsrl22/a09U9hHq8+BK+Ofzr6ED2KOxMKDk55OFc2Ue9/P357ZApBaPyxKK+K4PiIPwyRs8ffBwdHr+WBmtKYl1zLFh/4n/86AapwP2L9v7z4Phx0GyrZiFPXOySOnZZgZGUwJGyXKG7tbb1S6MP+Pyff+HejHuws0n94NseyMPRssD6Z+82Y2Xd7jz7hCr1x8RwZ0T3n2yLQ/Qg8gA/8W7ldWkd7iySPvcsmPPW92fz7rqT5/JXcPstZuVrlTRSbCgv1Fa9CLRprB39wF2XV8v44iDhlAuzW2ccofPG3v4U8ldadw288lMS8+AvHtgkDV8EbmlbxpaliTSQ7EyY2UfP1RSF2mCA43+A5SqYCfd4p3hDvSCKhELtLsX14yMNtIwhNcDd6VFm9WTZCC94/6RW4FKPgvCyX/bkCIllx+6I6puBLr+VlmVU8iO4UoeI2BqZD0T7zzAKEEKrGxgL4N3ydNcDxeELRg9Cd6cEBQVdAgRy/c0JW2BnCRNkzQgfRO+5xD3BxVG8dm32VNZflPBwFITmiiVi/6ypfpdUNHpu5EkI2AGwTE4kEg4rLFIMT4CEDvB5q9APTctkSy154PSLchd2hlB/JTrN+HHmFQLzJ8iAIiyG8GHh6WWvsdPKDbp3/nm56lBq63noUAp3WJznGLyQhIWmP7pO8TjKMeNwzpT5IePL/IBjm0J8L6M74fmfKGcltgNX1SdOOqqLWsRm+wT1XrfICHJh81hfGoQfBuirLBclMTK4xumm9u9eUeyxSFptiJnmSDIWSL9FkFBmhDTEA10Z6KDNzeZXe4HkoODfz4PyGkrS4a5bNMznaotZqFDz6Bu8NY4rkr2RtemdTBs1VGah8F5UdOgwwgYJBpZ9K0IQwKZIpe+f5cgkw9zA/7/Rvhzmd5c2za3EC1yDas+oTRegAMRImEtHEjSXzv9ik4M81WTbURPy/sA+cA78vrZ17wD9c2fXqOnSx+1Sa9U5J+R6JQmG7PgVgMW/CXEXR7UQxVEK8Hg4+Rx/sDz5T8Lsd75fq/UPdK78/R3x7BzPx230m98ygR+/1giZ2C5nfItPqGWxTHV9Mc3wBxfGl9IajNuzpzy6z2cffOvXu1tp1xtTzN2hIT//PVIzexIiEsq9xQEmbA08qW4JIXqfDN/iJICGL2Qdx8BpdkxqTghD/i6zinvSYDG+zCZ3ygfrY1gYHydLCbGLgOa/KdVJlaQ18HKkcY3/+Bp/q2Ok+287qQA0Ak6xSmvQpfRQKW1mLvsS8r78i+nAUVOeDDAzR7/uuHJ/sGnc8hX5vjRMtI43ZctYsbDiDUkFg8sj0wFs7m8aSSiKZVZSEWmXouK8xh4uAggZephf1GJ5zROLVwenk/kFpLEznSBRwHlFCv8xRtKw4MV7Ogs5PcZImB+BI9B1H4kVQGTD4NAF7yAXBzfs7InKwwDbBEYgCLFmhozZ9xwhbe5Ny3Lxzeg3+o/CZc+qF459xI4QcyhkjPwnNJHMNzFxxGFYt6n2QVbvlDUa7l5s5m45WdrJ/QBzF3hH3DEWNH8MuvhIjFohfoSwA+UfXdwAHpJ11UYnUF+56FwkK3av8wqH8r7IQ6TMynmJcTj+kpnWWVrNLlUIVD2hzx1vSwxS8MY/524fcTqdF2FLnMabz1w0oqsdCFU6B0mmiKunxrkMemEuPXMYdocaLR/chU4BFr7KhVeIlne0Hm6LerNeUAooso1KpR7LXzGFoq9o4dqKmalOpwxmhlp28KW0GjOjAlBOtnAdZQ16B3AWIjKRVQ2pTUu2AclXVybeAZyNLPAreCR65LWLbG40SF4Xyuun2zFgnsasLCip93WnZZg3MymquJIw9iClsmrJJl+q8c998hlK6ysQF2G9Fk0rSdVNu1eWpcbD+ZSgrjQm4kXNwvr68qSl5wzhklb5DcBVSnOAQOMY9c+X0SXAYpB/4EZEFS+LJDkFiyh3uENTMJkRmstEyRs1/zTSSeMunSZsIEz4o785AzyDkN2MbWbeRQVlsCnBMA4nWTTHxbSdy01yGo0AyCyXPsCkj5ELMlUrwBPk6ijsRH3N2CMAinBuNMpHnGFb73W2s9P2ofyFVNpHbFbcee8dOb1l9eRl3h1T046xJRBGIaFqksjAieECUNFV+za1lNY31emRvXM4eJBkV9VwdOQsVZL6FkVDrcBoP8yZbGZxBWTtJWszRN+wdPtLjcZrbs6ctoOCb4KmJUZsR1Y4lg+H1gRVm9iRrvrPbDuklxDce9MRgA2WRSKQXlazKAVTkkGtPQIcl3oS5OF8lTpdw2kGKHBUgHzUVMvSi/MjFQcDxyChk6Fbm8FNeYSpvws9Vak+6BCuBBBcOwu+G6ac0X9K9xEe901ml14lul1TpKllUKTlFMCFblxhbM1QaQ24wA0M+MbgxNNOJlQO+zAo6MLLYO2S2cDdgyyxmW4tpoZX13TO8Z9u48sjs1S6Nj3mgs++xH4BeW45BKAB9i+8HojnN4iMPMOu9CauXV0cus5m9kAmYTmoog6foRew2bzmIuiRpkwiH2L11OwuCMG87DzWuvLCWFq9ONiWuAawFZYonoKQ1sfoJgwlLHoDq5jjSRD4O1QdLrYSz9cZqxOKxrCJ2qtXzVTq7zPEs3mJdZHzoLdpOXt2Z8fFTZBUwI5qf8fJUNfLkINrxdgO+YZO1DxWX0/U1tIXM9sYdOrbbDFMTMwuJanjD3zHktDXYMe/AymK/H4QQuf/GWeua7zMD+TFFkzJXvY17jxbkEhf+1zl00LMko1F/60Awrw2ObRK4OXEbjOFbDi1ZlWYmMhmVRlGACFvEXkDKgJNaAp10TKGzM5hjwt4zHBqqfNImtxjQYqWG9nIiq97eiSrtZI+26JwRMDKcCQpHSewoBQ809cwGJeg5UNCY9C0BZDXQGSZu8Xeu4965xHJ4CUZwnljHEWxU8nmBIEAk9aSWe7MgFpIxToFB740bP+vIvJRVS0C2MIeiNljHpCad5jr7stvjxdjD3r5VXSw39SUnn1j5A/LcoVAPMei+b3ea9xAGNuU5RSzGSHCwQWappHLZYMCfoQs0CWfMOxnQfCiG5FHw4t1nn/CNVNkhCsAwp5sAHhjTIeZ8RG7owZiJDcMjYkbTIR7gNTybCP1vd0oeGuCGU6CnSHJ7wuS5nU3v56Sx3dHI3pgNpcyJnUikKPS5W4V3vEoiMYWAoLRFTBjs/8T0DNndU5d3Q0nAwLs7LCoeB4sQ4ezdMqDR/vP5nSpqYgRPjHvGwdhQKI/1GN62n5On/2XudZNEJ8EDpo7Mx7w4bV8paXXe2FWJBjzloXbjFRpwrN1Ya9ezAecGKrB8QCusvJngVnmNRaiiYeA0YAUhXo8FWIQZx5ZvNSJ3JiF2uTtuxGQosK/iBmp9zUgB+eG18kgIe7xCKDantThW7OPOp7YdQY/PPILbVVr9sVNfc5NhBr4pdQV6grdI+86k+nTtqJM5er+s/3LyfQd57cuP7IrPaW8rR0LuPzC/paVs3Fs+6LSBCRrXkqQwEGiFPfZK7bOdgK6PBhv6zFfH9mpQPUDM02fBTyeH915IcmtUAINsCsrv4IOYZ0lvQNbLG1QHC09v1yXse1Vs5/QZTUwhokliS1bjm2t9E05odtMH947GM3gjQ0QaYyxCgcSbUzZClc8zxTItNT5mN0IItS1a86W9FmfeZeTxCCHVWbDDxEBrcR7f6r225Atv//b43+K70M2/ZLQV59h2u13QgI83ywuUzjrfEpDJYLdjMQ5GB/EKVTOup+Rc0EfhoRr474HzgXNnx4sTQEeMAp+i6vQ4HJBrwGZ9xzfgUHhFQsOA5aut1CpuPHvEZmcskKd9+x2IMC+vChJ7VO5CE0IBiwddjmv190AXNfJcnLOnijtug1YEZsSjqlxmTScn3rw778sqp0IiEuLgFQIVwsHFFW8z/dQ+y1VPJZdilRb5AmbEz51s8NGDiVWwMvMQqqDLs31EUvkFQiiTv0UX4SLKTjOvPnZ3nU9wAsIChuvv0MmCoGmtcewZGQbOF+msO/pnD0eHCUQqa7w6xRIGQgI9Wl95IN/oUnjNQ35nOxsJym5BIvK2ZdiYpxs/cH4e2WHMEjwSDPB+0rY+7QdRF2o/MGPVcqTH0cD2iXfuJWXUoPHlnnqQdFWpwSJr3RMY3pQjwaZzALKp0PhOcnWpjPKz3CMfc6MlzkwkDuk8jbfDMOYt/Y0nnb6b9ZyijinGplUxyGFRXkWqHuRw08ziYV6XGChE09LOKdqBjXjlEiDivYwkC/JwFoJ3QAM0RUhDdsaMzQwhXBGJ819uio91ZEoHK6xPBBR1SAyk71ByIo99zR/a6oydXQ5ay8Wizoz7n1W24thv4D2bVFdUOCFY34nsOY+zgyAtaPNMFXPJ2zeey7HOHUZyPHBKMiTZAHrMSH+K2QXE2gJRvCMerWMTvGibuSpOEGA/kyMhuq2Kf7juqJxsopltltuRcgBOJo/VZeqJqsmroU7CnIO3hRcw4624SvmG/+kF8jHL1uOQKm6EsTHwDacxMYcxxxhuJnNQfwjOS++O9aXa7Nlw2lQLOvhxUi16TXQREOrQ1al+5smXS8gaHXmcEm/AfQ5chzkoyaqEGWGRiREfFXZa9+du2D4AhWpafONupbxdwylComS2uPClg0jiCCf5Wk2k0KNdem1LVTkZhGEj0uF0EFgvSMBJ5pA2Yt3Cch4wWvNythztG58MG/J7zj/qJp1MzdRLLB7FGxCLTlE/MmZqCaLTkzZDr+28gjVOjA1M9q1p6EqHs/0pA6TCVLoalYr3t2cAdZuuR0iprSbJd2O3VpyJgEVgNsy5mzSgek8mUa23SVuWSy8HICsDS002EoW4TzsjsaFOpWyQvpRVmS8SKSQK5jnf9dtay+SoDKwkQymahmMqmHvzrALNOeesAy7hDWxIPq0KcFkJgVpXdWqm3UM6s7lJG8oQbemjN4vyhLAsysWN+EJmgrumXV/CYpcgRneJLmFFO11r9nyD5UKNKnxyGUdN/SrD2uhm7TsT4NhMuFfpNVz+tHdOwJy31qSoaC1oU/OZk6DKeXIq5XngpNX68qG7e0vlQjv5zwNjfw36SrTKMagvL1ZqR8K6433VsogemVvPrrNrohObfGxOwEw15YBCVuO9JUyGBUF5I0Xh2mr3hH5ol6FUib33bBWnPKxeddaXUkeb8mlrujOhBaf42GgoOUmKJBk5HXHAqlZwiRV1YwvEmUoLobQikBsA08qG9FvR3NkKtKEh3g0AaJQXsJJZta7AhE3EkbO9opDRAy9BAl2eRIEz267wXT78zNQ4lYmmc9B8+qgb7vdBMfNDfMmC7hUKX0DJUHPt66kn7aduM36shBnebJjbw7vOeGXtP33HSO1HnYPhLBoslqpKWF+mT//wPPJUzMW6a51ldgvjogcE5gbWSqnHUThACTQC/8ktbSi7fniZXauihpp1VazQY+UNVeyRq8V2G3STPtuwoBEXIzlqB8mMyXGhaOPB1r1+kq3Kxg1yG/FkXSeXqtDSxUZx7pVWe2yn0se7oHtvTE8CXLr8fB1Fzob2BuRidssaTGPvlFTWG94tOK5V0m6lxtk6wNuBmKeT6esRKOTsaRotRbVbdKEVxl6xZRtjAkZtSyC+1aFSM/QoKjmEaWr0lsgwPudAInzrnZa1r0Tm5sUnWOKyuunNz6ZifD2YODAss8aJ6Ajd/IDcmJAJhw6cMHzl5GFF3qNhQ8q09YLN8ubGGSxYqEbz3aqh96Zjjf0zM+L3/XlXY58MMeK7D5IiA8tUUeVPZh8zKpFpEoV/KEDS96UJhcCpLkJo2KcOlLF7gZbvypqFeOX+OCWUjbddMVd7gu56eI5OupJMbmSrayQ0FAkzvvqNV5FX2Z+DTS3vhoL32IezHt26xSHoA+ae2obOsYMjroaCV2JWWJOiIw7hz8JO/TXrICbuvYLR8TP6Ll/czzjWAViXC42odMghEnbEJInFjtkaqcSqmr5V8GzbpQ3nfOpe46lzJK47k+9rR8t9R+MFrNrGKYEkV/CpILy37rIzrPcGyW5mW+cs/Pe8UmIfOraJpvoXp6zyL2qprHgcktUfVHbCyb7TMSlmwEV9Ug7pMlgN/hsr66Nbc7XzYzNuVKGnTJ64+JQBcQquTlZHZ2ISZ6tp8MgzrC4x24XYUyPLGoiAnTFy0+EqXUdundi4U8y2C9IwKYZ8gBHxI1UUCePPpJtjP3XMKgx+jLkEBGDcLe7ruMixHwBVHsbNsrUOdVtT2KiyhZEjLLvQXmbYkcwaaazyjcmf0f4u7GKiu71esy698Rl4K296bGXlRZxWI/F8/Bz7i0AqJzwvfP67+UfCfDQOFyrmW5HEetJ3Oggy1Jj1GIyeDKRZ6CtqLHW3u6XU/UOe/Q9eDKoXmOsi96breMrLUaSFv6ugpxHrVHR/6OWmXsqfbfv9MkFzsVkuI3PX64W/n416h2l/9GzKOezIQf395dfO6N44BTxkb+v16eEQ5NRBmyOWETyUDYaZ31cwlEu6loVsngVoNJBKv2ZVGcnOGAdSxczXXRWTBsJpSH1D9WtYx8IammVw3A14psehuNw2zO4Xdu6gUgyHxcmuI2lFp4W9rBud0rVAFF9j7kUPJMOJU6mKIuadQbaJ+dY/UnmOkpgnfNbBy5/4HHs1tmkf9By+tTbwUlkLjLQhRJUSIB7TmHVh2PnPFq0cN7PnSqoZe9PpR54mtmmXXGbXGMAKb1X9pf0nz6/vfLUiH3R3VUfzWioiXtYDNwznzFtW1spudNI7TSsX2brXUBXHyh/3bCN6lP7Qc3mwmxNiegQ+aEasamQGqnyNu2khEle/JzOE79MZ8Rfd0Xjo7WTFUbxl5PRG5TtAkYr0e/auXNuJ+y/MdMHfbeHONqYy6vC/lzUpzgCzEHyFd6wrQ71XeXwAf1vSjCeq/CBXdKtLaqWDuZwd7wjkt/m1LWgqt6p+mpiPGflHR2+M6Cq8kDObW98Ad49v2xHu8DfKZBffhV0/t5sZ0EkzoaXSryO6juNJKOj78c1tP7vq3yRMgDauvMZOMOd0kQEd+BBph5krIrEE6A5lXjpi/tblDLXbJFGhsVb2UprOLAmmTxZAqN50wz5mkcT26rykj6Lk5B/UkJqAuKvok+g+2y7zyCJVi6hzpGWN0znY8iSxGseR91wbMHOEPaKhWyfm0s4aBwTMbGF/EhD4IhSVV/dinPw23fTM9xPAU51FtC7XkWUfx+REW/N3fo3n2VNnHPO2D3n75jUf40dyffd7PvuOz33BZnsI/+0ZD82Y8HykN1XzsQ/0fidJa6YYS6THTRt3SwApQ1Dn15CuRBzppEO0EznUm5VXk6os9y2qVG0gKVOsAdEwLQB0aPAl6uYz1xZjdYiv+AzfVrGxNQLd2cEKGC/k52bpi063d4bF6wr8gzqM+1xqGivlzO1VDaH2B3oVcboiw8wh8JDsfkp1U0wk/QPNyoRuy2R1kqWzy0T91jA5BqNOwWbkIEwEFLTtWb9iQHpOdGcFlRJgtypV+on108BtOoGpP8ZOTQuTNXsqWyyQsFIhUv0YeucwWuUw+c6kWb4wIyAMaWoEGimUGpppaiSe/JUwKG4GDJWU1ZwKO/iOyFVzyXzAglP8ZBQ8MCTSFz+6swolyG9coctiHN6fWaf66Baus7lxOn92G2q68Qe8qke5KuRq4Yc7IylmIM+M5BgzvUXbytawRYqlHxcJ/VAtTnUbCaTtGjzj2U1owVG/ojySn1D2kZs9bGaibqt8dZ4u8Ve+TVy6zSx9lEhJDe9VE+vc1ZPIYsyuo/U9l1RU/Rk5EHc2i3rct0/U0RGep4Rcg5COVtxcF51VOdqWcWlXAAHbDmmBPzSFtMBAW9h3QNEeToT1qvyIFn0GCrk9weqANo+3Rh2YPVVu6u3+k1maN6OgJpPMvSpBdeBoL2wLMA6CbXHBQQcmDYyUenl48OPk6fne82/xd9GM3/AU0u/pwy1Su77yVQwIo3l7WL9uroMTGHNQl8BhQ1IpbTt/zCW3JTOUTO0vpm8XAm8reYdOYE5W227O9N97+5rLeVOWIBYMpEJ8bE+LSO6pNy9lf41vPaXxtyYbGRXHKE7jj/h36E7nA0gcqcb5oCyTu87YaBcgv7ZWQue+DWpYI2bxxaITd96RYH+JpuZzHcUM8rBzSSerMBaA1ZOyBd4nbtVh0YojTw3fbk9dIUkLH9AeTT+AAgdKxZZJ5jkerJ1vpJA3lWtvi4L3cLp1cE+30Om7VZPoC0RZ+vSAEtwdFeBPgPqM833z+H17EoPnFuXW9v1ar6eD7+7l1g5eIhhd3JOILe6MvrfoloO8N9bU0Qr3hUuNUKlOfuuqgd2ipJ8dIX1wdLQTGf39pMzDo51fNNL5W68Gxr9r8szvHPD7nGCfE+fTk1Axzz8HqrQb4ggmVVZV/BMcKXiDGWj4H9OLC/zNHomKeH9nS5JC1WY3LmmlVQ0CuroANsNrT/CJHg2P8FrMOp1l7c3EmpIGdYuD6gIcpwITCOENJpnOqnyN4mWcJPNyliSx2ZXOCFPpE4V7e5IaPlC1vsaSLP6Y0npMmdwDANvt4eWjFgTduNzaiZ08p5sEmGoa+p5RxUzewzoXbOcFdMhN14d3x6N+1iXA/b2qTbGXzx/UZZVe56vNau8SpECtkCW7qoWyP9y/B9umXO+xIbEHluaGb9z4QD1rYelS/gTS5LSW//BOBHNem26Nbfi6qNGhvTuIZ8SU60nfImwwtG5BYHolPlSX03qUd+eanXmdjRm9hWFD3pYfZt+g9Tff6Qdlu6y29bdkPdPpOHl6Wu4bCS2SqTO2U+e5vapUgoEneiKlVzq1OrrN9dNOl955q71BolhxvZ49lhZf5iQEL8BnyirrJ3WfoewhxoWZGFnwdHnRxu6+nPc275np0r1pakA1LoIOzPkOTJRgTZ7puLXSJ16Ky7ZNaNtimNT8QT+6WKfW0m75KHj2fB/2YLAnSwE7V7kAsnOhzXNoYS2dgNyVMbviQK/OEhO2m8u0CFzZE5uZ3hjtAO/CoEFZD7PiU17BilFA9f3b95PDt0eT5HRyevr2+Ch5PTl4TQ8m749f/TW0dXoHojMNTTNMBOUvA5A+1xFQYiB+ZgcGCCa2BPAftMPMDHmNeMdW+EZG82XsGkEm94K0qgXksNOA+W+gh1QZ1vwLxFvMjSCtsZWVkEx+eFPBXGdxtyzKH/+ggsfYzrgCdNuN8KiZSMzLuxP6va1ORCRRzrpqqG83n4kfP+31LdvfVmo7287n9Ez9jpWCcsfnTkUzfupUf95n9ZRjBSq6MJtQgn6SoLJKEpWVzxvj9AbTXifXeROxLgNY/wtZJVu+\", \"make_report.py\": \"eNq9Pf2P2zayvxfo/6AT0Aep1Xp3m7a4M+ripWnSF1zbBGl79x4MQ9DatFcXWVIleTfbvP3f38zwa0hRXm/Te+lHZJEcDofDmeHMkIrj+I24OpTVJvqh3F0P33/7Y7QXQ1eu+ywS79qqKOviqqzK4S4aiqtKwOui3kRFVUXbcnfoRB9tu2YfFd1Qbov10M/iOP74o48/ord5vj0MUCnPo3LfNt0AjetmKIayqXuspd92u7boemFe7Nbm8V99U5sfVbPblfXO/G568wi4Dtum25sXg9i327KyQIdyb38cDuVGYblu6kG8G6rySmOp3uyLutiJTlVri+Ga1XkNP1XJcNcCUrrgaX2XRc+AQkivLHo5iK4Ymi6LfizalpCnRoeuAmgzGrduCu8UHQye9WHf3kVFH9WtHSpMAbyBf9uNfdkfhrJSwPu3lSi6eqbmUsNPPv4ogj/Fen3oivVd3q+bTmTq5Q3guRN524l12cMEOaVXRVXUa7HJg23XVdH35bZc08zmncDeVNn20qkKM5hXTd+rn/tiGK7FbZ9DlW7diK16b7GAB6BlviUgeX9oGewwrqqF86pZ58Vhbd6lhkXX12L9tm3KetBE+vnJzwPWimDW9uU6RwbMNzANWdRfF59/+VUuuYqa35S/63Y7UeNMCyiuC0BYLg/s6OOPfnj1/ffP30QLzcCznRh+gEfRJfG+eCsUyWLA66laSchBV8X6LTTSzLRcItMBGkO3yqKfmlqsJPiN2MKoiw3hmiCjzok/0+jsG+THuaTCbTlcExvPmlbUiajXzQaQWcSHYXv21zhFjroG3qqEaiCpCSu4pmU4q5pik8gaqe0Z36oBwDTWMPCkOwDJyo5hsSnXwxIQzxCflYK/LloUDxsYo2oRnUexhBHjowN1hjjEsmXfHLq1gHYGRLk1zzPxruyHPkkjUcHiQhySnGYtz9MZTEpT3YgkxbUnYN5DXfK+FAUsgWXnjAJNtxHQb07rIK+LveiTvVztc73s5eCBz1ZEjwowxFeaFNQIxrPEhwgEGb3JIpCcNYy2G8RGg5yVINpgdFn0VtwtqmJ/tSkifDdH8Ak+LS9XabqSkIEwqj2W3hTVQaTUAT0ieA2XXgDgNPrLghBMuqLeiaQCZiH80jTlnFGUQN1/YKPnXdcAKwOXiirX4JBW0cvv+mh/6IfoSpBYhVXRHJTS+F10DbI8IzJ1YwkLyqLc3jGOzmCSB7Frurt5RARdq2Uyj0YL539piRC18WFuyGHa2LHoV9SV7UTNMc1r9F0JkmX4+cmbQ61aIop5XtblkOdJL6ptBiK8JMz8bklcoWxHZtdyPoHaqS0H1GSVWQ9iCdgApiHun8QRckMz6MJaDFWz9l7SsoZ+yzaJz2M+TcGp2sYva5jucgPiDtde9Oubl/PoPeBzHzOMcEyzq8P6rRgAbad/rxKI4m35zlby8LG1h+7OQ07Jz6tmaJ7YEvFuLdohekmFhDSKJ3gbHBrMCap3xYcEKip74KrfDiVKB+T3AhbSk/n5OY13Q5MJUwxij5gRIHOhtwOtAqNp+pmob8oOhB+I7CR++s+f8zfPv3/56idoBzBD5d89f/H01x9+MfU8Sq2rEqXOQg5Y/UxwnjPVL0mQhcIBOiE2Rj7UPLdpbmuSxZLnNqIfQOeg7mUSd8R8O6wD4BYcD0Q7N2WwhGHZ583Vv4A4fX7zOcd+2xxq5N4L/qpDwCRFDJCZehLJt8Q5C8ZFWfSaGGWxjd8zvrkfsyyCRlGmQAtJ32domdVDD7Rarvwm+AdEIg5w6KQcjP8u7uJVOq7XCbAXyxtUIdBkiSKO4ZPOV7MqxL1sqeLiM1AAWf0MDLHpUdWGFqKRNygK64MYl8LSKirAik0qqCUNfKK+UmWz/Vvg60T+6Be/dAeBZjxN6Vv6GRgKZwbNWKQsE2fegEpkeSTUYRoAJPnjs0V06Yg0pBMVBRfuC+jop2Z4gRW0aPqpiRQHKpigVZpbuXjfM6Tuz30e4ovk0NJIpBWnVsqkIgmtGGnHSYby2fW9bg2PJOpwyd7HTMwJFFxFd2cBGHj3s2Hfnr3HLcgM//cF2CLX4h1vTmZpf9gjL1ujk7QTlyfl74JELsnaYgA4MNX49pi85ROuiCSne+hkB1nkzLwzlCx6/m7oiqfdrl+8j38UQ7EphiIG3RFLROFRI39/73GJgeTJoGs0reSUB2UGLOKFg4UHF7gMbRtTZanlxA+i3g3XIABQlRK1YKGaalKkmDFk0fv7VL5TQ6FmejSBlSxZ+OUrpXZ+McNDpSrU1lZaJdG2ABpvDLRzwgb2XGbb5IsZTqB1095pAo3RmCKZYbhs3OYZQPyZ7FiYRgkAZs4BQbJz7k7/PWxjFb2kOQQyaRE/e/X6f2KvE280hMsfmHYzhvCUU/Hx6aYqf/ZUv6DhfOg0E2qVvz7HK3YkpkUlYKP5AStGWVfP6S+0G8Ydyj3r7LboarDmUfUeqo3SevsGdJ5dzYoIKOk+QdXs94z//KfvVEEZDdt/IA3Yob/DFvhQ232jFsnae7IcQFIJtf+dsPRXK2viE5oSGErGblAamXSIo5W1vqVdomrCNokjVSb1rd5hnqjTpM60de9KAbSkt3Ij7++39aZo3wwo4dnWw+DI9vTa1TUzEug7bd4mUmMtwLDbXQ+7q/2ZnKwzud03MzVFEVMhpc3xoY69isrmSPkIEGtjSxwfvKrsKGvuWoBKef8kN/szx7MAxk2NjpHNHIzpplJ8kBmzX76WXpAwz8w1oYvNHZnkTZX4Nv3PT/Jvf3329+e/IMnAGgmUv37z/MXL/461VdRf41rJJXLo3CDwwEAKXdyaYO/SP6FeOrzrQBh7YyzLIG3IeaG6kR6Kx7k8QJ+XtfR4pEvciKw82MtY4RivoBu0I0cV1ARqylNF/UNVRq8avFX+tcQ0ziJmmeL8lhtlwtsBTbiH4C33DSmRLmHgdkpJADTOUtfRoNwm7cgUBNRcblQuAVmCEEGmJbKLDM26sCWYMvfQutlflbUwTNyD2ur6YdJlgfYXDGtz3KdxnKNNX+hTon0U31BJBHQ/KWlTKFo5PGhATPGftbXVCDfJ4wxshlGoM8IFykNOGYaM7twSHPoDXbstxSYv6025Fn1ylw/AZPOobmf1pui6AizZ+rCXLjvRk+ssA330rtwf9upXL1CEwCMhb1taNYPbRgk5jb5emOYjekHbwvrSVAvYuA93rVhAIfTx1ReKQdegNgacNngP46KfCQLoqXPVetQYkC/rigygBRuY5vriFp2lEvS5epiBgZKk0acab1n1t0MzFKr/bdWA9oLGKXSP/SUesrLyMlGQv4ku0ug/okTDWMBvlARqR3h7DQJJtVGdfxMg2hqkbAl2mjBYFEPd1Ogu1JC/iS6ZRlFY2HZLIvgOQCejslV0huTgb9IVvDyG5dd/DMuvFaFPRNXDCvAcI4+o2i12V+9k58Bcm2YPhuG2OFRDDu8T5F6tj8BeXA/kelyqNY5rT7qrYYWId7gEJYNy3uHuo6YvKWw3HqxkSJxsBjDle9mmF+hMA6xm8APWY2LAZWSnL4B3yT2tB2wBrYCzcdGYFmmKCr6tCti6vChAgboONhrorGhbUW+QW9H3nUgUuNMCTOiIISEr2OLdGvZcFQJLjN7ogbJy9CA0URiRj0t36VSTvaZe/ABLmFYA7TZQvA1jI+j2M0HUxOpCa+uoQFzT9GATzynMKF8BWwz5FoBhvGlUQD753hF8Gi0WW/HCFBijyVxFMqmMgCSktD7+yAY2EGET2bAMBVVxktUYZliyb2Am8lZ0eUmmPtAh0fOk9ni6+vrQofHiVMRd2+XFxdGoxGtDYRum1raJcvJILDo02HMAF1upSVaHE56Cl8oGVtHVcRVVEDtQTnbOMbAnt8EVmJNGkBwKdLv8ihQ5o/7Xsqist02i66QzkDzSDpWvnnwuIcpVnaPqxcEpSsDgZMGsbu/U+HCRFWBdshZZxBQWY0OttSy6sJiBnZrbvC3Xbyt3Rd/B4miugihgAUMBf8o8hVKLYwyuYz5AMVC0M9+L/b5omfeEQQfdCQywiG8/i5leBeVXADko7AtvSHWzsaSOAaFQ1jr8+lC/BX661SzvLLZlzBa9rWr81CibadNqpfKFFIJu97Ypl9RgqLbQK0pUCeQzVnEMhslEi5XcCtGqUy8TR8rMStiZLQn8HPtbSVqYlbmAReQAZtPDm8mp0oxi+x/PwhQ0KD/014kv2Q2kYyJd1mTAFD/LoKPDzrTqbbQx9WoyThrXVMJMGqJyZ+NbpxbL8LpxeS0LsZTK1slhOed9sW8rgQwVrIqWgWE29ZdqDugFJlqhuSI/ZOK2WMYSx9ibTI69BaCsSLb+HWCBta5L5JhwA/vbQQyx02w2NLkqSDgkHD2YEFKoZLhRAMboMWFkEf/eDxst51VWiWdzjwaI3VA+TvIo23tf1OUWqAHg39uJjmUWQTyP4mtRbc6aw0Ckj/oWmJF7U2MaOuhmrLsRsMagN1AB5ZqxUiTpg5zfi+4G85FqnMKiklYZMjqOEc0dFzgyw/wIm/DKygqW4mr+CDZkMED9HopKg0BxpNo4taQFKM1ndEzLGYIZoLwEp6riVVmZHNacgUl4Ly9XvMWhh7UHUlbpfa2u55FkFFnz3p29B1kz1zV57oifQ5ToSpkL2BUoLg8flT0OlKm6KOgUzEwLIlniCkVlYop6fb0vOkw9Ms+50gfaGstcMmeRxwemoRE0FpSiJDefzpGz26q526OFZ6oeJaWplXmwXQq5ZZm1zkZkUob60pH8jnB3J8ajvdsRS82aJKI01iPXdo8mLfLpRCqdHql8vOjZrXZXTplKCTSTXAzr61wFEPXuy0yeKYz19kvjqO2F26LbH1pt3uiG8q21A6xNsxdFD+1xev1GrCjUEoxYhizYsBcY1dH9g/3/Jb1wOqC3x3cF3xoux8BEh7EbtSfoo0JtDW+E7JrcvuId1KnusDvs/Ax7P8dunK61Stl2mMe1iFxFOrcjYfvw3Np5aljcniN1ipAQYkAFGTMVlbMf5PetOOb5mjLWUFhQNZNjCQsABEaf72GZK0D0PMBeHaQnuRhXmfrPGOTYBIPTlJc6ey1faOOhFQUYpLKCLJmBgd50dzntTtIZlAUJxGnNqSRxkQbvglJ9Z7Cp3EqlIJz4BBvRSfU/aAI0vU7FbNoAP2HqBOXnnDr4fa/dJImL5dmIROgkhK4uAgNjULD7M3fEgYaaa7xmbAJDvVl+QU+Z/plNc0+Io7PRxkD1Sm5wss0P+0QjiBEvRENzLFhr9VBSQEdFD2TCZBb9No+ICXBXbeslplgLTeXsAO2gJS+KVj3RshT4CxOP6l0ynukZpeyKRCfsSgw//+LTTz93NBg3M4FAQwM6nlI0xvJ5riUpKcagIJ47Au6eG1JMUcyZlHYMufaQD9cYydEWo2HsAhZUL0M2OFZdK4suUtfqJMIg4Hx/hei4ZORVh9yyLo24/fICppIcTIQfvGQz5Am3Ly9A28Xt3748vcnfvkzvRwggi53Wu5GlJ3Zt6o/7JbY9oVfN3qf1aWv7PVboAV2DeWR7k4bwke6AX8NcYoBZZI4DA2wmgQEjNYfddXsY9BZEAqRV7jAqxj643XDuCgQOVIscyYJGHpkF6IxG2WH5jehwt4n7nN3VLNe/89zZgUnLjFXWxzcmGpD+YdVB6kzUhMWHyKqjMTPFuGD+UPKqeb8v1tdlLdxtlaQELNH86g5sfbV4lTq/KTvaxUmxCxKXaqd262S83If9QaYuMk93OBwnLe2HQnTWl2VDczI5xGlnnrWJ/Kc6R9VQ8DxMVUmgYt8Od6HwHndzoMl76KV+hzYYPumTxHHtcP/BRIRw2j9ouv7DrkEFYazoKXrkhCFxtrgzL9VxrOJd2S8uTciQE2BEPM8baMpcj9hmMyvkhoGol0XjgKjrjRw5aAxgeL4Mewt1p0qJOmhmduqcuD7IGEEni9ruA3haeTd4NcbXFFGhiEzEHt19oOZyPOikzY62M4+U6g+zvNaPbaefrO/L2vCuLf/YUKHaPl+Vtcz0HM9V6gcL3ejyX1M/7nhjYkcOcBmd9U5Q2AZqq+i8sTyuYIxSuDQFiRvNo5ukhSLBa6f8BOvgKplnfLiro+5tHwlpWTon2NwRGB/F2s/KZvhLKBOH/E6FZ9hI2+4Gz2DFttP1NCLBatq3p+riLKtXbhzYa4wrVyPKNvpmARzD0SyNIwhK+Iwux0IJtwJVfu9OvjM8f86NNPeXC+qS30C3zpoD7t3Giydj6SR8CY5W0ayD+a4SN5oWZNAAQ7qNYaHtJXp8b4GkBduha/LmpovZJgiMqjox3JJSgpH5KRWqihk7sCQVQeJ64BTbJkyMKYIv1N9+F+iwUUUqhWOy030JAzC9nbjOHMuwk0SYIEDbceTaLowJANGjnx45Su0jA2+7U8eNKDvDfrRgcJ3ko7WSeUszlE8hsyLCCkvxm9W1uXRj4iHkXJ5Ef1ySBDuaObeHL//kVAidA2F1c7uZfVcMxQt0HWkd7ce63NSCU2NeqJ42M8qQDIW+FJVlAA7q6khZ27SJDmil4YCW0UZyn63DK/qEKgVg9B6eFxpNbLfIZe3mpYbTOJ36KpNTmR6hvp3qSxb+qQ77GrVFTudxjQPZZomegMsopdQa7nSge6NiRg5GrBF5MwKEizN3MCzHJURmPIwQ7hJd3eEGhIqeZ0WN4+d1v9WxHG/UmXJ4h7hQD0MeegZW326B1m4YV18Z4TC3WxaHGpyU7qIEilwCTJP5LGkFRhJjbZgB15s2qZGxtnSnQA/4Q4pSXqgSu/nyTt5rVkSpyicniyQCc2p8PwK0pOJc+Tl0IP32WoAAZrh8E8Euj8Z9zlEkXyn6zAL42R+UkabEq6FH0UOHdAWATLvBRHExSHsr2XRNy73ZHF+2qx+hPj2uGTQLUnIJkg92oYSVTiOQ9s2l3JXaqinsWS8NF1Cs/HQ2UKH1aT7gu2yqbNiAfnE+UMWPZQTZTIeq6df9GOJS1QuyBEeMeEJS4dzB2OMKF1v2y+ULjt0j2cPBnqZTUjs4n6yyO6EqOcDqOPXEM3Ox0/nRA47XRZvLw5PGycJimEoATy5/u5W47fNeUOrqhVv0UH4XYRDI7OKZ72EPjkuASU/OIzw6HsTQWejiNqcgaHl10Dm+fvQpfIzZhR3ICmvpWgwJW8nzUVhjDDp84Brs0rqXK3mEsXSNHDttzQeHGYsDniYKV8c/S3cfN7ozY4QBDFb6vcIwA0MKOBGOIKyRGXUcJpZTZQareI/ej89P7M9tDcsek2hoDbldSw5jtrWbLaDeqRU+Hv8knk/IAHHLZCLPxUp7cXivp5NxgNXWt00vEo9/UDTBGvw8PYWeKqlovkK7KzmZBhNoBm7n+LWGNSVz9kHTDqXM4fr5v56+ltmo8+h9AKP70FUFRhp+JvnoqneHjs4p/PfscpWSgiM2Rnl0GRQWRip+pubB4QhQPD45xjBwuxhYws7PcavxHlL/wd12DgOTp+XlaM8pUGywzUYseAVbgbej+0Z+pIAGTcN8dOI4IE4xboBB4qkjw55KEZvD2uZ9hpNB5Z5tOQ/0tgotOBwnohAU99H5OfD0sTRQjywjDRyS8d44VkctA3NcVO9F1QB9IG5t//jxG1nbbH9VvuPQRJ9siCmjYgtaRa4RNovxMS04Mad/WTh5ifwYDEbZehkM8QjDc2Odqk6GrBx7oNaH5rAGcxUfci0Ecxa9nFW7Yw5lLPLKSye3c6XSGzziBpsdhmYPy2CdE0+oS+QKRAmhxKM5Zck5j0uwfGSS5WMTLUn66C2DlkrOroEqPHbToCHl2FzmY9Dv+xFUZXqG9w0WOdo2GKl5zvH2Nw4cYfvD3Ta4+D1248CQl/sGBBLeNpiq7q7hqujBmKh5iDm8KTS5VA5PhpKq/GCm19H2EmMq6mrDRB9uCeCRqRW/8O5T4yIcT7QpB+wilq7kjC5Hg8V7Q97YxYXxmrbAG1OnSkS3PwzmDkasaJ1gR04KPqgbNgIPq1G80ro2yYlKO6cVTtZ7HMs8Wq7M5XVoJjucfG/jiep95vqysMXvZZt4/B/wd3GZ3HTlTt0u4s7r0mnI0vdDiXpenqGin2+yHe1AHXFk05Bo3EaBuWBM/c9iU2P6reWR+xGjmv7/ZPa0Bp5imKUhvgq0yXADX0dnEs1QeC+QDfDQBGiCu3G88duxMclXT1hCj88cjGQ1z1pDoagJARWXbnDIkghbwlKbXjfO2QVQ+WGoUOADzaLNptkCW1CQSMuOb6JLGRW6mF083Ol9GpD1BoFHyvoRjZXIZ+/Dkn/U0FUAMiLkRSfHHsCcuWElmOx4fenCmnPnlpsTZrGyrWTKmouuM4OoxcadYTdGv7nhNXsdqGRMmCs5Yn0VqHPPH5l+nif+HG+qoJspZuv+ht9TRtocJCO8TgL2JrFXLg9SLuJPZl9t47FtpEwiz8M/so7Qdi26siftzLy/JoYTk11lPMErWCkd8ABLYGJOxnEr5XAEtkfjWJeqY24K1pgDx3AcbvSgpWOcrP0SQIlsmTBG+nQwVFPRGkzCYQRwxxXCzu3Gp/IyHpr28gLPN103G+XWxeWVsDq8/5XczTI3wGU6BRQshV7U/UEa+g92+s0ieuKDyqd4NbQUbTPGweylYmEP+sncrDh51PxBpnbOVvkK+4hh+EBcXK5vG/82t0tLQ82/B+nUU/rhI/quC/3YYaJTbmUN3chqbpfnl7LC3r1Tt8+THj/bCLQR8IIi71q1fnRLq2Ov0sXaU5dQOxFcfJFNH9xXgeBs+jD/Hzmcb6OrmuMtMqGY84NhawZMdXFd4v1Ed+523T1fp6o4YW11VbPTzMVtfKWzCYlbE5HsrskrsJ1UBBmbj0b3S7sGp39cwpslc2d7Tz6N4B0Os+GdzphQZjV2uruaqaB3okzHshILvGvKdvYnXz8xiV20aYB0dNNSUw/o09JHzKBCxA5gOIli6nYDvFgt4N5hlzMAm8ME5PJmgy52ssAeAmKuVwgDAcqojmRWNY4Bb5XSedOZSZemPHMFT9b9i75YCLPMRoT7Baoouqnh6svDCfA5AaXggBqILlYAHQypV3KMGze9SaEb8dzpV4jATq2HfXBf7Doh5AV806vkjkR+5mR+H82J15lTIU/vI26SSL0vJGTqswigAy8zduD9gQ8tJH5ab2ZGpDeObFmPtoWupRNO9aJsO5NeGMypPkYThajdYEgRdGzzRhXwnloLCCPzEhA7bO5feZVRqpIkFxr5hrY8o5CoCKWa3PH2EneJl5nMNoRxYSEjBSbjybeWKHrr5Q1q4uS0LVcFzEIyZcbGd4A91j7yWx87TE18GsY4+MEQJrFZ49ltB6Iwx6tDk2Czx7LoUHR45zlN+8JhgU25K0GRfzHFxnjdpPfJDIc4DOtjlNmXddPJ9KhSXe3W7TCCbz21annK7A6dP+wdTwGGxOMpRlt+6Pn6cuuBCF1yyu8GsCZD8Pi9F962dfSR6EAG4+m5kOkMfQuJ2rgsbIwlDSN75CKDMWYn3Gbwh24k+EO3EuhNxhjNY94sutMTOJWdbZWeItzaySd5Qb6xM3RNvD6Z38yo7EbNZnTtg/6ukOtw0d8GslnFzseCEnd5uknUow8MGSATnx46Ck0mpXNhLWGF05sfEBgP+j4DXRs1oLLK2SeJ/n2dSk1DHRqP77+hM5Ol/+H9mZT3h8a3XpuuRp+MOs4JStjmAaW/1IWr6Ra5NQv4wRTT8khnLn0mmujvYZma+sXY9BlT0h0qmkz+iQxmXC294xqrUFvvBMaouVMehKDPFYx7poKVdwwBq+nTE04Tc6oi1IIdlfAbmaJgT+VET2Pc5LbNSEdohHFR9xQuzJAQ/rF28z459RS79+2G8JliI9DRq2Jk9Xwk6J02KKv1LUxz54QoqxUyDPCUtFFmji+ay/8JS4NXOXqDD6+YjSG71tWo+JjSDJ9Xemp1C4e2tJprBUC/VQonmqw+VlYO6/yIfIsfujHax23uayfsVLZ5o9VGqIHSKYGuXlxONAEJhMD/qdZEqCKX5i7oZ8/GUEEOE7bahH2mpKpXzRW6LlinaRD1kRTFLn9odtEPTaAzI0Gdbp7++uzszatnEdHh7NU/3oznfCQugwA0fY7C8MQi614JnImux2Ln9ZszaBtpkei2c+QldqNqM3EYbBCWibqr8lhXpe7qUfLQhWVLEJAvB73Jd0pXHywB2X0g+qYKfcuFEXBL99qQ1XJ8g8ZqGqi8seIUoP5FG6vgxSEPoUmXi5yGowR3HEED7gHs5HUQx3CTF5CcgJkCdQQvC+ooVt4VJObGEAZr+paS1ZH7R45CGl1Rsnrg8hF78Qgb49E7SlbTN48wGLwAV1ZZb0UnMFjm3stjW/CL3aYv6WENeMHKNQSUip2wAXRpwCPF3XJLVW+Vav8UB/tY75TbNmwd8EvxUIkD6oGzpcoRkUXGS+B4ihz3wMj4AHNqOJ7iq237k1y89tJNdInmBP20G17dE+bY0D/8TTdncHy9uzPMWVB9DWIABt1wMAXDvd/ABq+Ae6HFgLearjwrzRuvtV8fTxvvqLeFluMVoMok5qinU/eubg9VRaoq1knDTjP3agl16vqhy1o/4B5WtdhgFDlilhNmczoa5a7Skc1tpyCLwrFE7c10CzMWHfX4XX8JfDHxDWTmoTXrivFkxpmLRUoy67zO2IK13TvRBXYEPQCUT1Y2DspkkbvbfiA4v/Td4I7nN7BRcQXTiLKfKkKxO0MFRgIsJcks+n/5LMAH3/f/+Mg8fQgBUyGPfRJhLJG9C2ida+aPC2g9jZ+qrmEGJvIrWFM2OfSNW8Bth9+YBhro79fPfkIp3hZr7fimtxh4NjWedrsDbrVfU0myEf26K+kTbIs834CMylPeFK8+wo6oTRKfnQFeZ4BXbD9ypRIPrkXVLmCXhN/ucj5wixFY+uyZ/Dbcufwg2Ln6SNLx3uSHkM6G5ow+T1vQpCzMYL5tmkoU9SvCv6ie6rvbZW7xQn239gj8/onuwnw46tG98EQLdXOT7IzPkp27PWwWEv9rRPqz7FdFX66fSSatxI2oFrrk5U8vXmWRsT2Sol/jNiTto09kTfrGFP6SD3N4Al7oix380FRGXPQnkg1i5itu/sfwsMJMLxVMs7FMKSfXfFCKR2jUp8nInbO235CnlQuaiwMNfRlPaq7xd9e8c0waEQKnPpeFN1c+Ua+gqfdFMAuAxTX0qttw3TFagaPPZ3nDz4KD5gnF6jARRc/i702neHLISzSKDjXmp9BnDOVHgYfEYEmfnjGyC/8p8QPgOON5jiZAnOfIX3kez3WCDXLbxx/9H+65kXw=\", \"model.py\": \"eNqlPdty3Max76ziPyBIqQRIWIiUbZW99rosK7JL50iySpLzcLb2oMDF7C4iLLDGhRTF8N/T3XMfDJZ07CQmiZnp6el791wShuHrcrvrf/35TfCPvM871gfrpu76dlj3ZVMnwQV8q8qaBYe8zfesZ22XBPCzLddJkNdFsM6r6iJff+rSMAxPT05PNm2zD7JsM/RDy7IsKPeHpu2hb930OQLtTk7Et+1a/vavrqnl71Wz3Zb1Vv65z/ud/L1l8re+3DM+1bqpKkbYdnKu39qCtaz4R7nueZ8C1rau8q5jqo/6dKq6MARqtNPfCU31pamZ6HgAfKryQvZ7B3+Klv76AHjLhuf1dRK8AOrkFxVAeZMfsDUJPrA/BlavGdJKdK2H/eE6yLugPqhvByAufIH/HgoBv/tUsbytU05+tZLNedatm5Yp4q93bP3p0JR1L7us87qpS2BVtsu7HXY8PXn926+/vnwfLCS90y3rX8OvrI2yrAZeZ1l8evLh3etXH7O3z9+8/ABdo7Bv87IOkyC8zKuyIH7iXz3r+hC6v3r78eX7t89fZy9+e/37m7d8TNbl+0PFsk0J/yoL7C8/tc2V/AJ0YhXCwP8QZ4KPOBmg9i4fOvYe6db1rIjeDzXy5GXbNm08Pz0J4B8Qvvd52bEiaOoKaLkBSQ3yoBhaJL9JkpbDAdIGNbsK/jffbqEDSEYHa5FCfHpSsE2QtSwvMpTNCLk+J2bHwexHZK6Y+KrsdyQTaXNgdQScbQpAeREO/Wb2bRgjB3fAy4qJAfhPy0A5apL6tGryIuI9Yj21IC/LekGDDPRyU24j/mMu5WkJupogOivC6y3IqZin3ASwXNF/GYKMZRdN0/VA86EuQuj/t0VwfnZmooUUDP6ZVwMnbhT+LNXfGR7sh64PLljAPufrHggOgJB3YmLoWamZQWar66zrG0IYJj4640vsHcjeQdkFm6a9KIuC1fhb0O+YMkrGjHKygl2WaxaucHHh+jCERyf7aADjS2qHGiQoePHudw/wcn+RVznobkb8otXQRKBdLAwAO9lzw3IyfwCa2yaz4/0oriYL5GRkbwXkQEEWrGhADIEfD3GGhwbuYHVtdoAqZWCVuGChAnZ3sWRMJYARsEvWXgcwHsSMuEIAg+5Qlb2cn5xGB0ZATr5vClZl/HO44p1QIUsw1xnJEnS+0ciEJHMo/mBcgXJBuL0o+jAxejQX/0JCXFLrfqj6koyH1YfsJkJpQaWg31l6dnZudkDxhk6XrIPWr6ymff45K9ih30HLzG4B+qEbyeAnDN5Ah6dn1rT5/qLIs+qcT+lrekpNI7Bb5E3fZJyao9GI0wXYYZjwm2/MBil3mzbnYjcPzu2hFzlZ+3v1YH/g1GYbVy/FDFQwuxnMLuBfAs/W0OFjOzCzfQcNzRaYnx1AJLOu/IJgzp9+a2OAIvrHkIOV/wJiAf0LDyxkGTbxnkgO5N35M7MPbxKSzsDcSz4ZwG75j33ZQaQBfqKzJfATu54HNyH7fAApY4iI/DVB2etYe0lfuUyjE41gSHyrIaDVgk+JGogKYwt9WvZs30WxHgSa60BE+yEhWJhDV438MUXeaOOiojnUzB5FIbgsmwq0o4DVanC3rlE3kAphYSALEH9lVzBTGMfBYjHVCx296HXc/At/AlYsaDaBPQU3gBY85YggZDX9Abo+EwuUln6H/hwsQ3AGuP6wCI77PtNWpcZ4NWUeHJquRNOD/iLgzUBQCE8kJsj7jgGbMaBCvkch/okBj1Qz+beruqqBTEwLS2/2/JtJQViqho+2HuYQVhempg8daEaPjkQQZKkGrBKkkwluQm4sUtyo8beaFDVCYhA8GisXXhvUKSSTnF0xTDZwTR3EogyMgPmt7LKhFj6P+vAQUXQQwn5oWcdq9BIdBLYQDKp5UkSg7bhTFEuNtUCIgcf145X0uTPlc3XeE+TgdtV0oCYCpNIRGZcDbsyWPt4Ci1quDJRk/78hhchzZRCLVw06L+HKMoaYhbdHBfUFOVeQnUKBdMMzCzqpkQk+9kaMOmYOVxD7IHUhKuXxX6fix6OYaRCpM94TQIZG+Ms2G+7UM1P0poPfhAJUkeLNSaoxHC4gA9SdBKp3hNZG6KRBorU4P7rUNyryCFRKLYw8EByyWpbDgvurJhAgRzESIhv5AyWrK4/leZCzgnHIMwPVWEZVlGLwIZywPxk5L8dU5Pw/D5Sf0DjuLAveMEe6mURDuzRqE5YL6G7TmxoprbOa6kMKqW3b5tciAJSWD00K9KwgSMCeopVT4ngfmiTbc6kw5wLarEzauSiabZQbzyH1aG20OpDifW626gw1e/W6Wb+qC/aZtUI8KHGEkLDssywCd7tJguaqhuYgfJ1/uX6Xt5B/9r8ImoVu2ob/4KiUBgF76SfOqIGDTcGAQcEvEQHiB0E7FCky9hc0XMG/8c8PrC3t2IBLh54oRd8VEaAEUlb0Qnu2wDDJysg9K9AJuJp0VpWfWIK5wSwvCujTURJ+WUK+3UDagMYYaA72SsBCDvSqgjSiok2ZRA3PirLlSXnCIcxVdWU5MhFgIxwhUn1RkIywsRn6w9Bnx3oDWZFnwB78Icb6WCkyMI7dHX42fCfJQmE/hWagjB1AsAkVxo6smAQBnMw/3Z44HrosydrgXzEFKfibiB36buUMsiiBFSPQvsj66CLkI6Ec6G0DFKYAAgkrVkeTcMkRqR52050kf1UDPHKIHLTEYkYBFcVyHdgMcEftiOqQTNTbfocLA5uWd2TToqUIPftlKJJri8CaCyCQBSZSCxgMY559PYLfbDZgaAV8kIY1eK0a/hdFy7MVGdL1sO+GfWShE8cuoLJq1gDENFY0JLa1DcYLk0KSXNrRErcYuDgLueXsfKXg/HRomwNr+2sNttvlB6aB9gPEdEgibpvHE0hGxskUTy2kqUKH5BSmEH/NhD2UUYBpD40JycShthSphIIqFtkiM9avJ+gEIs3HpZ4T4iQsBQLPE6xLD/u6W4xXoCdwpJzUCrFKxWAS7TEAr1Q/B+ffooMWki3sM2hWARa3aFhHlogSO7IQbbPGome9TbEOaUq3YAVHBcIWENIIhG1TNXn/1VOLAeQ2XC+kPcic8kEg8i951TGndErkhUQgR5tlpCnCC0UkJVw5MLOIbWrxkQ4peEKGWwFjnSSw8TGtg9x440Gkq8o1c00JdGkBPSxU4r/ZgSIx6A9QwbayLlKSHB/HEVK7LWiIC+8onh2738LFChxIIO9cLWfnx3G72rEW0kb18YfgLDH6PDa1VX222YTI1NeRDSQG/2l8+VHb7zj222wyW9JNCl/Ia4/wHavEYL6xYEDEDOORA0b4GkNnCrY/9NfcEihDEVn66/M7Blm1ZliYczXi0AEPqRTEP/457ZuMdn8iDyxTITtILzhPaFwU2QuaNJbxUSyl5UJ55dA7lrfrncitbUN/Pl8ZXAYhLQu2CFvKzmO73KUtIro8ADvUJbAsMudzmSBI7Ejgpsr7uqm/sLaJbGwXxiwO4cHdQaJpQlK/L91pVsHM9ramObfBSofBu2vPIx2+wGUkBcg7z7wLYWDROy8dlFfTgoGe5XC94Ab1botoyJ3TYmaJMkWIBLZnK2HPp8XfL+xqvIXZBAZ6dq12HIDjPu+Y35pbFxEqFM3txT7r8g2zB0fHgntyUyrD1PnNC2B5z9Dy044ul+ng+YcXr14FPIrN12t2wArvxbWqAjzsgv/58NtbkJq2hCQaJJnnOeREADEMxle6WigsdgN6VdZ5hfrDQBAY7lxEk8FtVw1bANSytBsuojZc/v/z2f/lsy9ns++y1WPaXw0Tilsk4DhO4c/yEEFLjOG3LECGBlRAL4U8itVFtAk3N9zDn31d3GY3OONy/t2zlVki5kYIYjEYGOu4HP8aFU/ceOVXVtMiC10/kXtdgrgtr3Byuod2nQPnMBNV4SPeN1e/ts1weJFDEj+XjPywz6sKI1PMtV6//50IX7B1UxgpKXqXLY7t0hMa+M5NWlu2p/QMN79o17mpZ0XZfVJ4Y5BFJUIqnKRBgDtpa0SFAF7wSpicWI7as33TXoMstVirewSoPtIkkYLaoX5CmJBD4rKRhODlLKqdbwZYIeUcEI+rUchn9hmPFmA/voMnEpO+gSyIAA31utkj4rib/nbYv7sOcOc+ldQ7mSpzHE2vA9yzAlJQIUKF5pQtm5bC6MUr9HelzJpbM+IWp2+AMK55/qZKjbJaH8Z/Or21B5hI8vqb8cXpux7aluH+1HVPvc8cUCWVF0ffNamMwyxLnj3xggemT1YxDYAYfaOY8+mnDqVvvWf9rinMvAkLkiyDoc1VxkUuUmmf5og4jHJN/eg0Sn6iWrfrVJy+ifSqDzzgP+QpTJSDQRbQacvP6ZdKNIZ6AGkzWo10kzoS+XDfusEkGHrGWgyxxC6zP3lKg6pPGUmEljZNrLm5v4eHVA6SppEaF2tsSKYK5foFb9JDc4hoZw/FODbFWAwoec5lS/mIx0uAgdzjg3weUrRMsCUVmSux5w/d6+92JZoOvfB9pIDvrQQymNo3gCY/IYSHZ+oCbRo45qHbsc6AR2azGg9Jg5eXIHOgZGBJGbdoaGHQMtbss2FMg64x4JHlrKSVgdS0LT/DACwRYp2wyg+dhvGcRJCcLxbUDyz/lOqUCOeHBB1yDZltqrarHZgvHSArRcW8w9Vnm01ZIgH7eI812Ai8TS+CsTGHbeWfLSS0tKYv1oCCVbLZTk6slWFF1hQ0o9UnYX411yLCqQna+kcqy7pAK6ENQm/3+YEKwVR3zS4GrIWJFfPCidKYyJZd+flYLSTBsyRyZ3Vh800TVEiGmf8Q5rQti1W2P5X/WKkKR00nKzwbn4Jm05ijtZwnFpSVnYKPYm6OOR8RmQNjHfRjvpNhnJ9hYCGoPV6KBVuvC0WJZlFfOP+lDzoqG0dNFF/xMQ/3WHZyZdx0d9Dp3N7CNlwhtAW0ba6/PcDzZdjixAPSDnZDX1YnVhM/3piW9aYZs8ATNnABfFB8L4wRmdzFg/R8E7wpf/5eWB/0RcbH9x8+qL/CZDSNsYSJRpt2T4Lo/Ozp18GjR8HTeDzCZOIdXTlF0ne8xhfFqdBlIgeobXcUwsgNC7af8FORnQjsMtAfngHzTbjtBa/+yY16d49G7rX1vg1c306gHUHymBnLl3RwiNw6/6gsTUZcm3uj/+RElR9pkJUnwI9ezhbBQlL5h6Hxo4jXE8He3zkbOzL9ztmzESVmvokgCsrjgWj3RPhpbTd43BA3v3eEtqq7prEArz/YvSGozHE3eOxA1MpQ8ngnv2/IxCqWsluqvUkZK/vOU2Ntn1Vv0YGP6OKVzwequSb3U0b6426w2PiOCqrxhPoYpyomcKHtE1XlR+DzcZHGly9v9G4hGiae6UD2mW9bBvaBjiVLKyepNQ9uFGMgbT9xRNuz/+MLRhWmzni9NrQL5NHsnQEZhk8BpoMM3hLkiGyJrJNYRchgFpy7q+IRCi+xEzJUaOfGg6rtx9MDXZm0og8ciIVDXrRX0YLctjkadRjFTr4aK/sj0zS0sHx01AjfauKxrOjwA1/AWGAkubSv9/CGA4lHo3k/mlro/5Tsqi+raSjN4b5AgsfBuQdQ1wx44E9A2pd1xPdI9ATjMZdYIUA9dzxDKlPF9GDniGMQXKEW1vQzQfhRZ87OpcXVuc3jxxwiBlEcu+UIisH6mcmFuY2D0TICsTqZAGgtZAJ/heqC4+rTUd7TtRx3HEiZUqy/m0VSkZmqUhVPUVt9iKpoBkxWwFGuS7yrgdzrUgcgH96XeWVdZBKVWfAjsoi+y6tLSlFh3q7EI4U8REpHxntqP3AsqXduCSrTOZY3jIJxCB6qm3tFY1z8Uum9NPS6zoflyG44YCTS0UGScjs0Qyey9zCe2gkw9jZG+5Kx68KFZvPkxaUbtv/gBuxy4GPD4U0MxBwAf/3xHr7R2B2EbvHJnzeGo2F/3QoKitIWrzcRvLdxwilsxV+NQ37b4Tz7OvGYtMM1z+SdcMWqpPsi4ti9E0UHlPiVrS76745juZbBurHinpf662elOK6mE8e47vhZHQIQ29Ge2j/Vvltv3lgDDfSVPzp+1uSuHEAXUpbyvpyxzxbrj57dQ0J9Yu9QXivIplKK0WEsviDrTgINvs/JFCNyJXR1GiHvHeDZapz+Ia77od5h0nxcSh8rnauFhXayxqE+5dusrpboc9ha1i+GsirkeVdTzvFEPgk7sCj4txD54zfz7KO2p+KsrXkURl5/FbuWmDBCJsrb2Wd0X8Er6kJkxGb4OtIR83IkHm8Dc1FVCuiPi6/Ts+SHb2SlVh6BhtiZro4CxFMrfQdK4urGa5dn3vO63LAO9cC8LSlHP1HH+GVH67CRdQxpGsT0aSXr4C/eNECdwdsyFNVzMY31DRz6GxV0YiILnJhI3s1RFxAa3A2bRpYIJHpZuHq3mayl6XuDQs/xZhmd31I66DkNTaDuudNt3YcwaEq3FPD8AJcF606jp6fnImR8z/uNSvdJ48WlI17lH2q6FlI8UddACn3vUN5TktQcFSnUDVCuaGAE+aJksYoPIIOWBN+ef/dUZO14s0nDsvccx+gL6KkH7NQ+o+PmnZ1Dvcd0B/4TYGAx53wetZypCf/02qYA3W+hZDrGdTi9YmcjwNiFWUzMbK+TAgWuaMuQJB/cHjgQedCCwjupGqF51tQsDtseK3yfX9Hm+UzsOWh4tImPW3ogpzDZ90aCAbbHfUeh4wY1PPHWndU2Pi/UPKAiTkd79SRQtAeXyyMKD4qZs7/tFJu1ACdT3E/cQ6e+Wxue4/3oS29uT+++y2H0kwJkdaUDNRT5UU99PY7iN7DJxlMDc+uEWkcbD9x9YCyEUd1qSeNWrjB14jsM8SxGGWoRoCYjGRyb19gNPdQER0JhAd/aE+YLRStJryW423WCZBr8uEAwVXl3/4GwQV+S4KiMVnpEZEYwx9niKNg9GVch5OXYjC4Lwb/wFO5K3I4z70f6BMAI3Aww6XDAm1uR49uVW9cnDS1eySN5EpS+PWgCj2WoJW7pCg+tNwSt6CA2LvIpuEY4fNd5p40n2FD78TcSym1QNMaxbTWRyGZu5Ifb8P4EFSG8ozDkv/lRFqlp/PL5Cn4BKdEqF//JIP+Ght1Ol6mPRojydtzRG4G8BOvwRxAE5snaHGsV1jMIjpuV3Si+BZ9KN5/i0/GVOBSL7UUqrLx1NUlqsHgkZZVwNi2ELOrPpiouPCbHuIuEWxTbpqW3W0SfxRJBmPgu5F/yItLpxJ29+yBvPOwyWoHd1rINa3HUwqKQgf2dC/1L6xMX8w75Nb6h4jwcYc0TzoMpAocetKD7ca4o6JReY/+JOF60m7fLQs+SPRC8vVb2ExKGuAMA62/7cQOR31o5qMF9i30OM80ahiNNi/GnZOSPF/KXxM3hhVRNiIsaOCK+hzV3KJFFmYWPTtrULPiP0Xe6/7mwH1KSt8w9SzAujbqDbLGNtUzLssMej35mm/OM3+aOvFeb5ctSy6WOweRdaH5MkKIuKgYmZPpWMgLmdOeJpe3lzGvE1jUfhi4WPS9IKYZ0eEZ9Hpgzm7eCjZteIyQM5wHifpFflOAcSubcWzGmGdV+7J4yaQErzoOwaKpgpxJfY1bAv9zTkRP3Po2Dmz1K3p8x6JUEs/M4/XhkKhqCXlYHE1hcNGl+13XF8QH+Tfh7rSIW/fpNoAnIb+DhawUjZG6/D0IfSAXwZhrVW2ekeauEz00hFFW/t5DaRdb8SZB/LruFef2I3jDDSxHiOTMxbaLBJbbh4AIMkC5Zm2/ZIiS9Ad9Nh6eK8pJ2iRZn44sCodSwUMhmRDPG3O2f2vVwKfrm6fJXPR5VB+jv2Zqw0JcEntNx+RltwKicUD6SJ14Lg4SygSV5Xiez7kBLBf1GVJh5zS4r5eTyoON97kzrP+kNnvZa3KG3r8Ob/kW8iJaVBZU5jZYeOArapvDgm+mGmpqvLc2F9hshjVptJh+FcCEYXXZN82luWDu6XuhFPeDnzGk6cyEFZGlYQsrUkVWBk7jEbbqW/LN4nyLrgU6ZoAGhN+4tKUQ7MtPkAKDlfthLYLCioe3GZMEdtkyweF/WQ8+cTpMvBQiOgjiI35x2zUlKvuQfTi+Xq6I+5X527xZbzEb1JX2yvrpDPAIga3jjliODUTTw2LT9xek/5j4+tTH6eDo+4e8RBCpEeFsmSO6IhtpJ9LbGYyTGgqNI7G11QXiESgHwtI2Ym3d9BuZqA4NwVyHF3zNK4lgbeXtjN1jXHrdb5Ruaad1cRfIZzXTo13Fadg1MDFltpC+PmzcVDIsm3rSJhCHGtfKbXhmaAWOPUf2i9xe1jeBmYXzf/Ma54xfSFgLW/JdnqxgvOsk/z+HPUKgk//J05VyCo1Nq0EK3trzo6v63ttlGF6HOT7D6UgdSjsoDLe/FEuz33zDDODjS8peWUGQBo5SCbHVkVYltFHu2EgPjSQZhlWI84GPFR3IKf9lkYk9rEwJBZsaRBuVlDQusIxhVSUl09eRGTGxtLJqvN+FGvxI7WLmfl1ZggXGAnYJSQKBwAskRsyZOF22Rw7lrsN2+iqHcdsgBtuZND4LIBIZYkuF2dp9KHBv5EXjKF+33q+aSnLy64usSu3DwcaijUIx08w5Q5nz8iasjCIk3sKbR8UIYITMBRswkA04/HrLVj8TEWBODSQBK/sBPgM4UneA/WZCZY+XdwYZn1aPtpyj5JF22wTul4mnZubYdC39skVhmUCuReBNRqyLeO/Bs33J1kzuzaEqgPSWV6DDEjiRlxUvFGT5OiFsqAkJHRtW+o+zOfh9LNA7wCwiysF4LGDJ8GFBClaybBzfuRLejh2WEwZRXb/lqj/lmydG7HLKl+eYDA/gRbNq+7KX988RNxm02fkcMvYlqjvR9qmPhFYmImsr32MpU7GVMb4+iZ8iFxM2Ox18/Gke5JuaJvc+zkHxjsHmHZJv57I4VQ0WuUI56cCQAVgeJaDBEcxUeE8KXdhe0bo4BrjUyuEWH5BRF3fcAJAb4uhINx90BA/T4TZN+QLULcWfrmnaZWoZHTkKEx0HQ4xVRSOML+m4hS81hO9T8WIk9wdig6GjUE9zrqCOxFCMRmI6fWkBFWfrs1spOFMTnU+f06MvP62oozIT8yYevgu66Xu9aEGOxBcwfe9/hY5BV1Vzh7q98ptgFqOTiYUfZPj+Liq+YIox8vR7AedCld1rmTJ7H4brlUY57R+R/PSp3jmW6ku9L48fFJFGcsVIgzvD6kO5ZXkdLCqU9jmplhdMm/5dzgcxq5YoA1UmqZv2Jbxr3uxTscBX5M9wnkyvxQAX6lXgYoMj6pidbMF7bo4lU+rFP8h8JRMdTDfRensrstKpgHQ1faz6WHz4Kvnp2dqYMoScBfBQ8Ozu6QAFQzZgYlHTJ8GQCWx9jpo9d6OMXaMxm52co3PSWY3651YRcPEi/2nQmMYmE4rODWnZ+dka388yWT/Tcv1rh4kERJn5URtxNPExM3EkTDxk9E4zV64hZ1uHHxP8hwib8IHwkt8r0Gob6f0HQYmjlPf8Bh9CMVA==\", \"train.py\": \"eNq9O2tv3DiS3wPkP/AEGCvNdisOsru36BkNkE08s1nkYSSzh1sYhqBusd1a69EjUo57fP7vV8U3Kbbt3N1eECTdZLFYVSzWk50kyS9j1fSE7yjZNre0Ji9PT5fjMPU1eXP+d/K+udrxn//ygawrRtump+Rrw3dkpGzqqnVLyWZHN9f7oek5y58/e/7sl13DSDfUE8zBIO15M/RV2x7IZug5bMVIP5Cu4vt24G2zJk23H0bOyDASHOJNfwWgNQVsSZIgyu04dKQstxOfRlqWagWp+n7gFaJnCKVHx6t9NTJqBv7Jht58aYerK9jAfB+Y+bhvK74dxs4MsN3Em9Z+PVhY3nR2g2lqakVkXXGKc5pE/V1N7yu+syyTc/iqZvhhj3yridf9YUE+VPu9INVs1E/d/kAqEODeUl31NYzA331tB5lP+XVLq7E3orRHpjd8Y0Y+VH11RccFqfjQNZsSpVfWsPGCbKp+6JtN1Za7imnC4aRpq9Gkz58R+POO01Gcy2e6GcYasMlxoWnA0nk1MfqZ/jpRxmmtJtdT09YlCAz0jDM12FWbcSi3L8uO8rHZqNGbqm1QsiVXCEtQrW1zBdMZMvn82ftPP/989pkU+rzzK8rfw0c6pmXZVx1oEUCev/77l7O35dl/vvulfPPp7RnA//sfJYKabmFtVcstFP4Uz29FGB/Jf4nDy8jyR1I3G34BYws8tsuVJFEuKHEBYEVYsTiTs+IKOSD5sKd9SntQeyC2SCa+Xf45yfBUd3C+LVVY8U+z9VayaQvXNt+A1LZDW6cZKQqS5HhsibPK0gTk4GSO3KUSe2bhaMtosIyPh2BEkCFP/FB1rT9Jbzd0z8k7MX82jnCtgQ0YjSAB2TJKPk893hEBmyb/eP3hvSJ1kloExubXqQGLQ84POPs9+duXTx9JT2ktbAm9hUMiNQUZ1iDDAwhOqCbseUQASHXOqi0t51IA+YJZIQ0DS8WrfkNTpVzioDOHC0n9f1TtpGnXCh7SPwDCbmKcrClYLTKs/0k3PFEbKu5qoOvOIk/244BQQluTBUnERTPf6O2ejiC0npfj0IohBvLA/2t602xgxMEFhqNcDwNDYDDsAkE1toeS8UEYGRxpunXVIsOlkIca3dJKWF0w/kANcOMhhntcgm1Xt2QcvjJLKljhqhPf1aXGj9b0SJIZEyjh4yv8V11Xvce9sgINgPV4cAx0itapFlleN9stHak9pcyeolr10IFtkw8KNY+f3DU9sBW5U6ju9ZEdM0CaCH2wILleYbRmRTFd1rRCKWvKV9rgW1sizMsWVJSDvfk49PpmdtVt08GR7oZpZCAVAaLQXBihXl4kHmByqejCQy/XYC7gWLumnzh9EEkE3KBCMXvE/FCQU1fiUgRIuxycGAYNoE2wUY3b+su/I6/+dHqan5JllMrvyJ9g0mwdIAv3nl1PxVQeQ62vZwswEAzBLdXQvhAV4/rwHE58aiQYmCY6QvRjDhvgBpbT/qYZATV4pTQ5f3d+9v7dx7Pyy9mXL+8+fSzfnr1+KwbOzj+9+WtiRT3D5nAbIQhYS8PhBQo8BSEu1IHPcILSifAmx3/SLPOVWcx0A1hIDAXA2/x+trPVdHNNRLQIABCt0VQrIXyO6PyCjFNfNrVwswsijYgIONSIMUhgSrrKneEQ+lG+wqgTAgG8O/BJ+2O0OywwsXKnZEVSQY08DjWYaUIy19455ATL3JnMo9tDEKE+QBSDyKJse4gl82WjQy/ECuy7mGcgC7J8mQFuhJOTWeZb3m3VtNMo7MMd2EIwhcmwZnS8oSg2/VH5ow0Xo/rjPYELhhZ0QVILqWcz2FWdSg4UdQx0CVRcA5J/KywicwE0OQ/b9DdOeMtgm83QQajUYLIiAi+OGQocLnCjERrLvplGcCcc+DXCu0jUoCO5Sz9YOEXbo5f+ULgCfTqpGjm5U5jukfxh4qypKTnN8zuJ8j4J/IuEttduBAWRXkl4e5bO6Bd3RF+YYDi4OAYZcdmya9Bc4PhsDyshi+KHRwy0I46qHcGuHEQ4iXEe5qfaN4N6SDKs0AKZmC2tVErtdxsMkoFHsBkrb9FWO4jyTudteT98TTNwGONWmMPfnfzjpDuplyd/Pflw8uV32X15h9lfjv/8AQB39PZi9efL+8TZVxl7EatBGlNhNJS2mFZfrbvyho5MiF6YMAzU+A45Z/YwoglGR7thPMChyFwvhz34BGZcjqe+OLy48sB3wKLaF+6rzntzfyb1DZ+cw6vUtHSMLNNT/rqQT1gYDrngKlN1oNVIXuqxsgwi2/3BAe/3xyBlouzyXR8FVax5XKoPPnub/eQBjcMGVAhUOZPlDDUOaewOIz1fNMOVSKYBBwgPciDEJA/TDKUKqPhlnKh/ILsDe/rynyrI63zKzV0qHaVLhNalzojvZAYOO4JnK9cHDAMltFS5XEw6/sNcgK9jgzHApNNoFVyrQADH62ZciSR54WbPsehAzkvv+ti8Exsob3YsdJDTbbWGtKWTOAPkwOdlgAVTMYbqzMTt1NMy+XkEKH79VdTCp31LL4Q4hFCCmgIIC669Eht5oY8yCYHy7hr+TUEWYHmY0CB0v0BJOVxLhbInYFJjNDZBGmOm82mPhjGNxlCKoBw5dnUmzFRXJNGlRETLYfDBXHWFBUkPoZ+5rohQbhciksnCrj0kIUk0FjOpbQwqkuKuiBBmBNV2rDSml7lH9boSme2TIOivMOvNCRsoPDrIHyfz09OXxwNT59vjwWdkNLZI25iW9qmn3dH4Vsw4yMV3FzByTQA8MnoMu76osEro7G/N3qdrEcOWZaGubdqKMbE5cubZAA/WceMA+zSvLkNreRoXiWtnL22k7VwzVS50LvoLecPU/RNVPc/ERVe49Rd3TVjSTS3qhdk/OwIrhCxx2vzGWaAjL4XGA3GcwWYAf70fKZql2kpOjyg34HsFYRWFCf3/MIdAYYOE2A0B/8WlMv7g1lGRMHuBpA+oli4fa8xC0qrcBSwOWwiHzCCrOrDqoFl9s6WMm3FP4eSomzOwYRo3WDXQAgIOcX+vHowpiATMBUOQSwXFVhlq/wQEfRz4T2hXdQbyWVcf9QZLJJ9UI2+2YKkwBdGFNHInNzEJiKw7MN70MnEJ9NCnU3ZTcjx/FEwqcS1cBA5aeQg5iIX2dTqHAWKHkZdGPKLCXpaIuSyzHFK6ob2BgCuX5+3ohbhSYrV7NRS+B7hRiyJXMeTMo20RQe0pmubxKJgpIyK0c5NQ5bcNlm7kUjiiVNcHRQ9nFWnrWFujSyzHozDhhRENtqPkEAR9XBtUNpsQusxEEA6x9jhWBx3SAKpWqaTfTpA9pOqaKiZ0X4HeQFqIpSPJobBoyLnpMkTm00DnJUsLf1QxFYx6fMXmJGvBDETW7braXBdt1a3riki7twGyriAgXulzyNmh35T6RqVS+IsA2vU12ayAiuZJVxRFe+VM/IeRo4WVTa/8qwwVAmkkP6GgtNxg4QujNaKsQuvviSNLGXyVEHrl/BZbrbXb6FX5NXo34ChZzDpAZdNvhyKIkuZsicjNqrQIsVLI7EGHdBs3/4hue19taFjR0xHrvE2HGHIvhgUjKQaN4xE2Gi92w4T5dKrrFvmF6VxcXiSzpQn6hThWf9OuulWVmBJLT7oQcWTrx1YFVe54IWW5hOWyi8+WuHypl+s6935gDW9uaJLNufY7CDEqLO9HAFSao7qvEPA3mzfycOx2Lb2hbXEFIQDnY6pgF2iUTBtC94OAEgENcRMkIXusMrgXBpPsihfJSVqxDRZqMkZOUrECnZD4Jj+s4BPoE4N7mTGtuFnMMukHAyqkwyZme7X2LuGDPc5Yb/MdthTb1iD9sfgD5AA//JHIZoTpQnkdTIlymPh+wpxo4NrdCfmrcVC6zDP+ADSz/o7sHXSF89mRqe03Fu6JOG3IS/cIrHVQt7DQK5yW36WzgL0KAdmrEID22FKpywHi6bGpqbAo5oZAmoxX37FLJR9K9ko0scW18s9XGl7Rc5J2WUUJpZyQ8lRFf88rukvkSOqBmdhMHczTrIdarKw2Gsw0ORdgaJPpLVjW9kCAC/sG561EKKJQWm12UmFeqD4LGnV0VoTt28a0l9cTNreBNP+BhYm4F8SzlKqwKou9mpEwJ7/0dS2vh6+9OAuv0ePJCKepK3wBHwFsthoW7GNolsMiM+A7tdNgf7hwvCJ098umbhAvtukbLl2dftCBewGQ7VXin8fyFZOpOGb0kYJXGJ8QnYHJk8pl3hR8lWm5Hns4ZbennnvZxWy5SpPVaCz3RpN3pEqaeeZXRDOYFKkOfTpPyLL8qh3WafKdynLCHOUp0ZLBpeyw/1pFaNNCq4EKDTBmfkh/HmpSBog1Ld94MupORUVn1RBfN/DRk5s1bCJraSumYDEmcx23Z0M+Ix9oQgAVOWHSk2B1vqUcb5Xh/QQfgmieYv2beQfnaEdJMymeH7nxiYq38TBAlf4iv6VK1yDsLL6R6cyLlNBIqR3yGU0QI2AbUbWFYyHTa8boiKDKP3+hYwPq8BuIScTCRNFL6oHKYE29YTTmGQJkK1HmnolR5+rGz1KMFqlMREsuspRyYUVQ/SZmFiaiM1bCrVNJUxLBYaL7Uh89HMEDOWOwt1UMY6YMvb6qs4sEnQ64bs/omOGo2h/hcG4gBb+ag0Tq4owx4e+PSyVKgX9l5E35ig8pVd/RIK/4sSsTkbnIak71zbEdR5S923/03efRc576Xj7DOtZddaIu9aDC93rYt7/X5GCK5SzYDcN16uX4ZpsHHemCSDqF37RPlJwL1g8QDUByPCMvGi4q4+qOOK+DnEtkIaKKUii5BabbSL2wH+M1gSJeG1DCKALhBFCOMygi/sEHjniJ4smuPezAF3JgMXdZEyvkf3G3HTB0sXx56Ubr+hhErjcfnl9m2R3S6MpqCxS6h2ZczfzsfUqebgoVe4/6AQfj/DmrcJA3zW865buiPa6kZdNvwPMBLmMvmVuDwj8PwaaRB6/RmtS/spYUnPkD5aNHS0iijPTO8mmf68sCEhEn7lnL79235jKKYDLiSiIyeOwUI0uO1Zri9aaYFusiq3ypDqo+e72ezjKMYm4DHrMzj99ar8dXyNd5niW5CNqAx/Jv/NXFCEIu3BTOzcIvEg2i4rmjqNBHFMH3ReTRoXkRWBx53eriP1IyKmblJ/ls7VgJKotIH+zB6Ap5Hqd6ZIh3nXqteN9ZPPktrbv7/D1p8Y3Paf1KhTYG6AIvtGqqSAq76UMdJOhunUz3ICWg+1RNLf3RC9DNXroRgXE64CttqTiVCwv5XxYtl/lxvqzkxjyk7xxD1yaquGDrNZQcUDWLADgoShTfkKQEmIRBwm1ZcRHdWEe2ttCipy6jqETyXEAEjFiwy2cXJpezWACvavBDlxR70bEcfmYEbQJZ2I8BzDWle/tUXUc7EYtpdKEwn+ahg3Ih8R/zzOugXqB9wr4nPb3l+sKSr03byh+TxSJtwLQQbXkdGJDfk5dzsx7+jkcbdZnpuMn+Yx7GvjQNlpo8Et91eRTFMsxodhkcu/2xiOQCZIevU4eJH0kuVyZOvQuouzc1j+LOpe0+mR2eXKleDfwPU9T/bXrqacQbU5iQvz40PzP0tAEksYIMDTBZBrxO8Lcluv+iJPdh+Tjq8X+a2Ppt4lPbThP9sxKr2vLB67yltjJvSZgw3Qbi9Xg1YZB3Lmaw/74ZGxEwFmVZDxvx8zm7NK/qGjcSa7D7pMp02N3fVlPLC1W4eyGskCoAPojBe46wxIK5RYYl2odXy5ZGsCyRo+wFyJ89sj2ALDFI/YY9j3Tc8LQOe1qIF9dPxyabG0unHLbkw1L8Sko+ICswkhjxJ0gTDV4/K5Tu+VutQCcp9cEprkuD9eUAut+d3TY8lQ7cxZApHKCp+ieU4qeGZYkYyzIxP05C/M+f/Td14Evx\", \"viz.py\": \"eNrlPGtv20iS3wPkP/RxsRgyQzOSEyeOsQrgOM5ucHnB8S3mTtARbbElcU2RHJJyrPH6v19V9YPNh2zZ8czu3HgGCtnsrq6ud3UX6TjO3wSPElGWrFqIMi53Crhfs4u4XPEk/oVXcZaWbJYV+Jx9iOeL6q9vPrIzXookTkXgOM7jR48fzYpsycJwtqpWhQhDFi/zrKgYT9OskjCwl2pd8ipPsiqJz7CxvgtWpXCdw/nc8aze86m5/EeZpTaUhbkpF6sqThQeOTwBcBqJL9SRnlTrPE7n+sFhuvbZEU8SfpYIn33kOT712Vfx80qkU9GLcpCv8YrxkuVJZTqkq2W+xsY0N205TyNowZ6RQsCCM82SrCg1Lh+y+aesWKpu5XkieJEGS1EV8dR04qupz/JCTOMSSBrCBSAfTlfFBaBfZFN5iWg/fnT0+cPnk/DL4Yfj09NjNmLu40cM/pw/DQYvd9/sOj5cvt3bOx4M6HIweHX88hldHh29fHX4ki6PX7x6hx302L0Xb54fv1ID8I8uXx4+02D23j47fPVGddiH/3AscPPD+0/H4dfT//5w/BVxcXawy478DfD3AHn+8fDkP49PZI8MW0v8+V/8eYs/F/jzBX9+wp+/4M9r/HmCow9/ev81fPf502n49f3/4JKHg8ePTt+ffjhutg5xop/Co/86+ftx+OXz+0+nOOMerAZgFFU849MKpeKMT8/hgRaQ8RgFyWdlVUx89ilLxURSOhIzFqLahCifLkrfAQmdx3Zeo5AdSPJ9i6sFyWaQ5SJ1QcKyCORt5Kyq2c6+46GgLEBkEqEG4F8hQKNSkvwgyXjkyh6eNfU0SytxWbnFKg2juLDmrlY54B3F02oMWPuIC6CexGXVatStcm3UTD9xWk0UMggdZprFc6CJtVw1K3vKHPnYwcu6d4C9kD0IZAGTZMV6IwQl7wRC9W2MX0oV3QYD4JlIQjWgAWSa8LIMU74UJQAa4wUZOLzwGdiulJWgbSJy9ei4EsvS9Xx2LtajhC/PIs6w7QAJ5OLVeDjxvImEP4tTnoTQWpDlgzmW/NJ1sStobFZEY8c8dCYezS0f4NRq2TAZMJevkmo0UHgrUXAt4TBU9utGBcBqsRZstQKD3RrA2FlmERAMewFSG/sFc1G5xN44As1TtA9wmGePahFBPbHFtoqrRLjoSA6ksBEC6lpOoG5asIjqIKCr2Sy+pC5AYschiYcbJa95kc0L9Gv4jMWzLlvQQAyYSEpgv8P+ycB4FiKt6i6jq9aYa0fCBm0rOACmYVcSkWuaRF5LoI7T4NvMucLFXuMQWipdyYXiZWuuESB3pRdxfUVTXjsW/cpqnYiQX8aliz8H6I6Cw0vgMZsXSLuzLEsAydNiJYg2aLIUcXBAUMXT8zDnBV9KCCPnLKsWwFRSnTL+RYyaJlXJIcordUFxJUggE+ElgpMjXc8yYNQUlNCjUKGA+3zP63u+4CBYGGBoRQV60krqznI2aHNxVWi0UkF0GElnwpN8wUeD4LktaNhJUgt0MBKXSn6WHCyzJtI7DhwjKjUN40GDhVc1Jg65b+eANfzsmCZgf2YJGPjGE29iKYdj8AYAlmtsDLfavQnSAhGWkmUPkWtiT5/2zOn1gbLxWPLiXOAqlOdtIKDa2pO3u+52IV6IYg1AB8FwtzsbChY8fBbstinyLY5A/g7YMHi2p55dW3yEUDKercm/grTrII28MjitKa/EHOyeshlT5cAPWMel/5M0oa0SsEYzxjJjIOs4IYq6nLh+RqZVDSGsaiRs8Sv5hQjBckJg7GoHMZfa+o5a/drBGvet2tCsygXJ+woDERgbBW95xd8V6LGUU7ttvSDk+I/uXl6EShlsDcBgDelCwQAR9sCgDJiiMbW8rWpUZk7Hqo0u2qE3oATLc3gMJENzW46kKgtQ7SrMzunWa4Dcun+O0XsezXxaH/FtZFB/ihYYCXodQD/H730QzeCBXon1AODVawDLCVYqTPg6W1WuV7cjq+Ffl/CI8nj0bDDw2dlZdgnEnkJ+NXIq28A1xiDaPV0JHR4Bu0dXzhFEPWhCgfuoQ8RQ5nzMIqvh2rNkJaiyEHB3NT0wpAOujwz/gQwQVlYhCDrkJiPnz8GLmUYPRXSaZJCUAYKqbT7FxCURU7NwrZeupj6ELkY0akVs9zYYYXctJp3uyvCOu5y1g+8o+5aWfAnBrvuEFwVfg31I8wCyL7yxQuG60WdBEGjpBms3J1lBsycBjAeT2gup538ZsXbi0I3UaSIXJuIlQXIveIIOGC0JXZLXpDn0BCn4HNItGLVKY7BrOB4sYpnzqXBBhBQCO2zod1AASYO0VoxgCPi1F8+9Juk24DNWs042IKZpi9lqSMkoBMIyvyybmYa/te3pNy2hb0JWvKbQSAeAfk8w3c53bHGHx7ZtdHUwrSny8youRAS9bE9eB+OYcUK0hbk1RbcOxFxxGi4hCo/DJJuDLlBGCuRqN1rw7EGiKCBKaAwxTaofnxZZOBuaTubeeEAyhTHgRZmPyk30YoIons1Ega7QlSoP+rlapqVXy68abAsrj8GV/x3Zfoz4uLNGvgVR2fS8ZJrvO8R34IVIIlCtKwXv2mRUkJTOZTZVT4H5jkQoTrKpDBcmzdSn6WdlQAFCWCA0d0hiL0F4VldYTw9Yi28T9h+jThfUnVY3CdIkbWBg+CUtAu1euTpD0S8RjWfkKyggdocvfbYX7GqEcp5CUFRvr+CfewexAeONDTs3NMjQ1N8E/yYJO5wC4/h03bomj9kHcKMoAlZ4vfOufd1AzsoPMFD3mSvhgkD6aGLkBZHMZ2sK/NEfQbRYecj3X+Lc5ZTASLLaaYSERHYKyT0MBsBRYvLYTEKBqoQnQ9X28xocYtMLTKG5AZR+2spIUFKkoDbk229grfKqkXOKjUC7J0/s1GRgy/jNUGvkDUzQ5DjSRqwJeOj16ZnPapsKpBfpaom3wlW67LUCXcKIX14gYLdO6xmlQSPnTy/oz+mmZCawHwGZDb4nooQJHUllVE/IxlUuTp6iRQpMD+V+gZKeGfgAUsjW/l7fwEua03XeZFlZ0favZfANoN5UtwFHSqyrBXfLkYmYizRyTe9XVod2Dm+HhatcLlit2/mgbDE7Ih/s3OYpvVuJRK5du03tOvK123k8VvaBaxMyMUrT7WSboEkXEsruLXDahkxDwfQL/b+dUJHZVoEAWKZWoAJEqgF3Ykvl8DA+7k9a7F3IOgFoxNUWDB1ay0TqhrDahMD24K2i4CdEA9/GfNIO1YqwhPQhWoHM/O7CtPG4GYwZdqJlciZNVx13XLVx0vvBXsNL32KkZVtrMmOqjOJR+0az/RDmpm1qOnNvObzfcuycABD2VUnHAxiQDeZLSeuNelpLqaMY0BD6hkgbjGBByweWamsH/bcX72auUS+zFAA8omAQiDQ9zzOIpU2rVgM1GrUAl+XixmWAp4fnYl1qQbfjXbBZ1NHTIqtOICEbUt3usJtpRxJqdDOO0Oh519tq7XCISvuioVBnvLhJZ7s0m+iQZCxXN6ZAqZXl9hAHxoloLlQ4c4b5TyN4GQS7xiL0aLZJhAy7iuxbqf3qWE3Yx032mg0mtv5OeQVg3Rao1vo7T/tA4x437riO6PRUR2rm4LYc7e6ZNR2Z8ewMkqZzvSLQ4YLPUaTJobmbCR8sBU9BuNSGiUrcZEw3CAbaNiwhSJCJLJ6KQUSIhzA7HeOjZJzPhJ7BoGAfWikWOpDaAbh4uVqGi2xVlHiu9oQ9eyFh3zQMVDMPzwTIhwiXcbqqhBr8YtDUszAVIqLNA6w8CKYiTtx6OU8MpZ42sCZy6Ec8jZpLeq2D3sGv6ES+GnNyL/fx3lg9RN+Sk1Owxrd6EWhwIB+IgWZAu5pemqqjqxZ9r53bHY8dWNOWKURjPJ0uQL5diCdBU4cABMR45KzyXBQgj7PKXv/+d3uwplO63YnJM9gI7G8Rn63oGOyhwzP43z7pvbcPW/I0ngHPtjlll7uuoR7SOGdXFtDaErpyyjyJK+eA0b/o3xBhuJdn7wB5leJj3DvSMMdqEOiqXB51AiUdU/uETqTleAjGrpu5LnVBk6/2ONTWRp0rO6DvleOZAgDsa1GxsUfU69xxlWrBl3ILl8v9K7SBFiTt1sidQEdwKHvbeka0lEO5DdwACVZnELzcBVk3XtN2y3rttVO+hQp20m92R2S+gxtp2m1KdmDKbthIfMMII6RCKLedBaMfv2Q/MtdsxiHyRAuf6b0MdSs9EoEOpA3ytPfqCU+2cNx7HcsIThZNW5LNnU74jufYpXvZUKZNMf4Rie/dLTJpTclcQIARLt59rTNhwN5aduUBAvsbNy7ubzO7NvBGuxmq6kE0LzDislPmIJvtox6fpZhwJ4CvKoBoH/OSpEofDmODcsFzMdYBGJ4B0/PXbLjXOdpR4QDAhvEvG52VF3+l7MSiEOUiSyITsiCGPEU1lvNClPSU7cpzdYkHQm2GSlQWlH2rN8RpNq91OC03+2/oZHTZLHoMUH01cNLsiN6A6lquaMhB8Hwm61pqskossSgIjbQ8TPJ69grJschJsDYRZsQWyLL4yJkK8Du4Q31h3xhJwx8Tr35bgOASDnIVry3yyjIbqfMdhwtOakXhPa162nNspXs0Jegh/PB3emASLYWcFFE80iCPIs2RdAXKr9x0rOMcaTiK+UyCk4dGJaNq082lcgX/1vZ3NWK8xHNH15w76g0wtXh1DDWyDSiVQMIAURQlajYkrRdxJEZOPAcJwzAqTskxmRZ7dbUQ1nv3feiQzgFCoGMWGVdLWeU0xDI+kUfxsrRrB/qAS7UNpVdzOz2Qzyk4GPDMeVbG6Uxep2Kurjug28TsAXkrDZUBQmNC+1y1JFA4sLfvbRtWSD0jeDhyf98cGS5lykeKDHRaZN/6UJ2CyEA2jDeoy5A9jQb4L79EIvMyF9Nq5PBVldkFD6TXGA/QLIjiyHIjvUbfZz2zW7xrO+86AFMqojSy44/v1l17/S+FwM0WYOf0fv4fUb/r2KbbP8m+7XyqxarW8o9EtV9hc0+f9akyEJFSRBndsiHfssKXYc1Ix+ZqbXb9ujZpg3rmtHvQU4ekcxN7Qqr16YKIZncBEc0UCPmaAR2TzOJ6h/02CNaqVe1Rcznew0GnAqbmStvHDs25O71vqt4xbA/AaWF0eBsw8srfaY2WvDwnQQNzvOSBvA3xnZVQ/LziSe2SMAPcbMHkQG22LuIihlBUUWuk3r9wyYoNlRWjvIuq4C23h42eh0biQS2ctQhb0f6Apg1ijrY9Yy7wh33VmdJvZNu0kN/JwEHIhHXx/NsNJq2OrkLQFV5uPn7s2KL6HBJG61NIA0iHD5bhNGpfd9rurNGQwa8RbZ82mneOOsH1OqwAh2ZkvQ7zIjv7t4u2cQfnoPetGNy9UhkSFp1LE2TX5YFx2B0MhsZQYb0MBJqy4y+iyMowic+Fi6MbnXQaOtj6cGSAsd7L2iama1WvqGK/1+xFnTfKTFvtA+ltrXobSA6xImvJrfAMSEZv59j1gvKZh9s9FlivGf7vNwu0GvACMKkujW81oyFtZarAwWav5uNu9akpxAbWxulK2GtCaQunalPOrAZbxwe+vZiJVUNJycOzXbtmJgd5qOhHZ56ksVr23QbCvpnYAlFOIZdBd7Sauhqc9ThCBaN2uoKHKK1WVWvPmPpMuzta76Hp8mXmXhEGlM97nTPshrSgZNkJPWqHNoRX7f1amBVu5B7HJZqVympYU4MmmWmuvGuSUZgLiFXpwq8+Ktt41Mr1IzE0xl2DnDTLZz30sfUMRgwtardlrGaZ38tsv4dBNbiuTLa1CRNeDFSCbIUne13d8lvbxspXd3QsKMARJfU0mwW8R6Cbg79TqJcxcWOzWN9VpLcT54847Y4+TnOvLDSMbHfOOTdXpO1aHqAr487Sns357cT9AQW0K5y052hU46CtMJJRtbI9tXt3tNHiv1REA6TXUrX6WHzlLb7yzXztrTQMevl6E2/5d/FW7gzVnNVsbVOhI91jjBrA3+h/9aqe019zVQet8sm6AuGEp1G2ZH9dQRLkbIrlKaZlX7IyruILwU7uVELUiOfvCaMZ139Oxc5FuXOCJ4snn4+2rWPEA2R64V7ajXJ0hRZSGc4D/3dyXFwHyliT2Dk/3HxunBd/kAD7/1sIDGlXK94FfH8X8a75wITP5BcmWjFC/wcotoiCyTVpkHRjQLWCA93JPN8c+PbD05ZSxql3CHdzHkUiChsLRnKCXQDrYbtc6QK2jZQlmk0PY7C9ZwzRoZLfRd9e241xxY1c7wHs38TO33t0/KuowPeL/51Evxks3y8wvqc2bBtP/6uV4rdWiK4ydIK2E0mSO8dpXwzZHiA8M8B2JDp/rFjNhFx3DdVEocr81OZqO2QzHVpv8D9IjPYdRetZEQnM+gx+Ab5jqg5a3bFTAqWhwTH1ehM8+5gCi/DzRWP5Pj7+TsCMChQtcq5gpLLc3oped+rjaGatjQuBL7zL0rjndykax43EF/IgXAKkA6Tn+55dFJfNZoCZPlJXRlIZRSoQdHckApgL0muO9ILjABiOO72WXcJvUFk3KI+uGVpo/a01ebdRonHGqeReV8Ut3DX70SBH6I9VaY7PNFRp0G2MW5Vw1HaHSjhNFEDAr8vi0Xwiei08el8FJDHCjV1pjNVL7j9i+dJg38e3upAuAI0+DgOBLzzTd3JZLtVAocm4smHImqNNpUH7G865wHKPARBo6PgHktEfJtcQwo+u6KtL2KqE+IcJGiLPoeWHvq6wogXjd54KVHLXm3Ttc7ykHf9gsLfJdn+dyhqWO5puWcLnKgTvXQn4RRQ7EtS7IVDWiChcG1H8Ny0MrC0naR4xY7OdnQlOXxmUH8Xj+OL75i+obP5iSvvbKiR+oZR5u11y126547dW6C1GG3Cn169k16ssN/W7tkW3V2pbcnkIGiwEj9xnA2+rMZZ53zaDt8w1YCiNNUTg2lgTuajwOMvHNu3onWy7gV5IaDTguz2waPMecfNNogXON3aU/NBrRHBvrwyaLgHgSOLQb2sH29hZlN6cFr27wVpcbvUaccdYvFPI389IoMyDi84TPL90QnRTzPHqKut/2RuBMkveXAusD6Px4ziFWIJfAMRgISChVpgldetB3gvMMoxEsKm9maXOx2uf+KT/GykAwVqK3+lvv6h7S9/2G5C93Sc91JIMVOU8bStpUauRmeOUjT3ERkK94Wl/Ma3KIjfEvbpUR5tx+Z429FJfYpUbhfaQyd0t8F35a1e2/aZM7itYlkPqyqDNo+1KDBplMdO3eXcTEGuz+f4wulkQwTLtN9Kg5y2pB9INOobPBX2gzP76TzeaCOf4iswBc525elfmNANzw9QdGQ3/Zgj6NSdXviwjT88Q0ld6J0ffdkABkZYr+ZVECxwBwrcrw0iA9eOlsL/CwuzGsorqTg3QWPfdEzrVsPlZGWIngg0NDBrAKeIr4387/EIbztZrGzbq13WWJe24jBZ8pn0cuVP6xAtxQH9P1corOjVXmyI9yd+OzaCvuE60H2lNX9dhSUSaltxM/fjR/wHHOUgc\"}")
expected_hashes = json.loads("{\"checkpoint.py\": \"8842aac9d9cb53226d9aac743f44dfa2a54dbe374864685c323b5ddef905f8f1\", \"config/data.json\": \"539f22e52924ea692966133cc618a9f53aaf22829559b30fba986c88634bf01f\", \"config/data.smoke.json\": \"425c05a0d4a012f302fbc254e63872b50729b7b37a5f94216e02ad881fb5d9c6\", \"config/orchestration.json\": \"be2079aa98e416c7a1d46fc037d9ea347b0491c9253a0bc0783f89da46de1d20\", \"config/report.json\": \"847057bd82f6113f47ac60da728b970fbd8b97b6bf9a8015a087992568e66bc3\", \"config/train.json\": \"4a622edca113728dcaefb66f8752d78aedb77c911102d2022bdd4de8e34b79f0\", \"config/train.smoke.json\": \"dc718af201d2b1feef16bb10a4d3aa18ebb5486af61c725dea8de0a5bccc1874\", \"data.py\": \"2ec61a62d9d9ecb22a8ae2987fdaee0b13d9eca5ed6f625168528ac7130b0181\", \"make_report.py\": \"746537d648ac74aac26d7ef9e04291cd5034bb202c1cdb4c43f3139d89209f95\", \"model.py\": \"c5dc36c062d2259c3d10310897c1d9e8469247294ece1d07defbe8ca8cf83c3e\", \"train.py\": \"9f3f29ad50b5edb6cbabf683f4dec29f6046a0931238a45b45895998a0c6fe6f\", \"viz.py\": \"2f6feda08ffe0d1abf476c291a33e6d86dd0761f021de440967628da0f64714e\"}")
for relative, encoded_content in encoded_files.items():
    destination = SOURCE_DIR / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    payload = zlib.decompress(base64.b64decode(encoded_content))
    payload.decode("utf-8")
    observed_hash = hashlib.sha256(payload).hexdigest()
    if observed_hash != expected_hashes[relative]:
        raise RuntimeError(f"Embedded source integrity failure: {relative}")
    destination.write_bytes(payload)
print(f"Extracted {len(encoded_files)} versioned source/config files")


In [ ]:
PRESIGNED_CONFIG_ZLIB_B64 = ''
if PRESIGNED_CONFIG_ZLIB_B64:
    presigned_path = PROJECT_DIR / "s3_presigned_config.json"
    presigned_path.write_bytes(zlib.decompress(base64.b64decode(PRESIGNED_CONFIG_ZLIB_B64)))
    presigned = json.loads(presigned_path.read_text(encoding="utf-8"))
    os.environ["S3_PRESIGNED_CONFIG_PATH"] = str(presigned_path)
    os.environ["S3_BUCKET"] = presigned["bucket"]
    os.environ["S3_PREFIX"] = presigned["s3_prefix"]
    os.environ["RUN_ID"] = presigned["run_id"]
    os.environ["AWS_REGION"] = presigned["aws_region"]
    os.environ["AWS_DEFAULT_REGION"] = presigned["aws_region"]
    print("Loaded short-lived object-scoped S3 operations; no AWS key is embedded.")
else:
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()
    aliases = {
        "AWS_ACCESS_KEY_ID": ("AWS_ACCESS_KEY_ID",),
        "AWS_SECRET_ACCESS_KEY": ("AWS_SECRET_ACCESS_KEY",),
        "AWS_REGION": ("AWS_REGION", "AWS_DEFAULT_REGION"),
        "AWS_DEFAULT_REGION": ("AWS_DEFAULT_REGION", "AWS_REGION"),
        "S3_BUCKET": ("S3_BUCKET",),
        "S3_PREFIX": ("S3_PREFIX",),
    }
    missing = []
    for environment_name, candidates in aliases.items():
        value = None
        for candidate in candidates:
            try:
                value = client.get_secret(candidate)
            except Exception:
                value = None
            if value:
                break
        if value:
            os.environ[environment_name] = value
        else:
            missing.append("/".join(candidates))
    if missing:
        raise RuntimeError("Missing S3 configuration: " + ", ".join(sorted(set(missing))))
os.environ["PYTHONHASHSEED"] = "2026"
print("S3 environment configured; credential values were not printed.")


In [ ]:
required = {
    "lightgbm": "lightgbm>=4.0,<5",
    "boto3": "boto3>=1.34,<2",
    "requests": "requests>=2.31,<3",
}
missing = []
for module, requirement in required.items():
    try:
        imported = __import__(module)
        if module == "lightgbm" and not str(imported.__version__).startswith("4."):
            missing.append(requirement)
    except ImportError:
        missing.append(requirement)
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
import lightgbm as lgb
import psutil
host_ram_gib = psutil.virtual_memory().total / (1024 ** 3)
print(f"LightGBM={lgb.__version__}; LightGBM device=CPU; Kaggle accelerator=none; RAM={host_ram_gib:.1f} GiB")


In [ ]:
preferred = Path("/kaggle/input/cicddos2019-parquet-per-classes")
if preferred.exists():
    data_dir = preferred
else:
    parquet_files = sorted(Path("/kaggle/input").rglob("*.parquet"))
    if not parquet_files:
        raise FileNotFoundError("No Parquet files found in attached Kaggle inputs")
    data_dir = Path(os.path.commonpath([str(path.parent) for path in parquet_files]))
print(f"Preparing deterministic leakage-safe splits from {data_dir}")
data_command = [
    sys.executable, str(SOURCE_DIR / "data.py"),
    "--config", str(SOURCE_DIR / "config/data.json"),
    "--data-dir", str(data_dir),
    "--output-dir", str(PREPARED_DIR),
]
if os.environ.get("RUN_ID"):
    data_command.extend([
        "--s3-config", str(SOURCE_DIR / "config/train.json"),
        "--run-id", os.environ["RUN_ID"],
        "--maximum-hours", "12",
        "--stop-before-minutes", "30",
    ])
data_result = subprocess.run(data_command, cwd=SOURCE_DIR, check=False)
if data_result.returncode not in (0, 75):
    raise subprocess.CalledProcessError(data_result.returncode, data_command)
PREPROCESSING_PAUSED = data_result.returncode == 75
if PREPROCESSING_PAUSED:
    print("Preprocessing paused after a durable source-file checkpoint; training is deferred to the next session.")


In [ ]:
if PREPROCESSING_PAUSED:
    print("Skipping training in this session because preprocessing will resume first.")
else:
    train_command = [
    sys.executable, str(SOURCE_DIR / "train.py"),
    "--config", str(SOURCE_DIR / "config/train.json"),
    "--prepared-data-dir", str(PREPARED_DIR),
    "--output-dir", str(RUNS_DIR),
    "--upload-checkpoints-to-s3",
    ]

    if os.environ.get("RUN_ID"):
        train_command.extend(["--run-id", os.environ["RUN_ID"]])
    result = subprocess.run(train_command, cwd=SOURCE_DIR, check=False)
    if result.returncode not in (0, 75):
        raise subprocess.CalledProcessError(result.returncode, train_command)
    if result.returncode == 75:
        print("Session paused only after a verified checkpoint; the watchdog may launch the next session.")
    else:
        print("Training reached iteration 100 and final reporting completed or remains durably retryable.")


In [ ]:
active_path = RUNS_DIR / "active_run.json"
if active_path.exists():
    active = json.loads(active_path.read_text(encoding="utf-8"))
    print(json.dumps({
        "run_id": active.get("run_id"),
        "status": active.get("status"),
        "current_iteration": active.get("current_iteration"),
    }, indent=2))
else:
    print("No active run pointer was created.")
